#### AER - Access Entitlement Review Reporter (2026-04-01)

This notebook authenticates to Microsoft Graph, scans AER review workbooks in SharePoint, builds completion dashboards, exports report files, and prepares reviewer reminder emails.

In [ ]:
# === Cell 1: Setup & Authentication ===
import os
import sys
import io
import time
import logging
from datetime import datetime
from dotenv import load_dotenv
from msal import PublicClientApplication
import requests

load_dotenv()

# === Logging ===
today_str = datetime.now().strftime('%Y-%m-%d')
hour_str = datetime.now().strftime('%H')
log_dir = f"output/{today_str}/report/logs"
os.makedirs(log_dir, exist_ok=True)

log_file = f"{log_dir}/aer_{today_str}_{hour_str}00.log"

logger = logging.getLogger("aer")
logger.handlers.clear()
logger.setLevel(logging.INFO)

def _get_console_stream():
    s = getattr(sys, "stdout", None)
    try:
        if s and hasattr(s, "reconfigure"):
            s.reconfigure(encoding="utf-8")
            if hasattr(sys.stderr, "reconfigure"):
                sys.stderr.reconfigure(encoding="utf-8")
            return s
    except Exception:
        pass
    return sys.stdout

formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
ch = logging.StreamHandler(_get_console_stream())
ch.setFormatter(formatter)
logger.addHandler(ch)

fh = logging.FileHandler(log_file, encoding="utf-8", mode='a')
fh.setFormatter(formatter)
logger.addHandler(fh)

# === Azure AD Config ===
TENANT_ID = os.getenv("AZURE_TENANT_ID")
CLIENT_ID = os.getenv("AZURE_CLIENT_ID")
SHAREPOINT_HOST = os.getenv("SHAREPOINT_HOST", "davidshih.sharepoint.com")
SITE_NAME = os.getenv("SITE_NAME", "aer")
APP_NAME = "2026 Entitlement Review"
BASE_PATH = APP_NAME
SENDER_EMAIL = os.getenv("SENDER_EMAIL")

SCOPES = [
    "User.Read.All", "Mail.Send", "Mail.Read",
    "Files.Read.All", "Sites.Read.All"
]
GRAPH_TIMEOUT = 30
GRAPH_MAX_RETRIES = 3
GRAPH_BACKOFF_BASE = 2.0

app = PublicClientApplication(CLIENT_ID, authority=f"https://login.microsoftonline.com/{TENANT_ID}")


def _refresh_graph_headers_silently() -> bool:
    global headers
    try:
        accounts = app.get_accounts()
        if not accounts:
            return False
        result = app.acquire_token_silent(scopes=SCOPES, account=accounts[0])
        if result and "access_token" in result:
            headers = {"Authorization": f"Bearer {result['access_token']}"}
            logger.info("🔄 Token refreshed silently")
            return True
    except Exception as exc:
        logger.warning(f"Silent token refresh failed: {exc}")
    return False


def _graph_request(method: str, url: str, headers=None, json_payload=None, data=None, timeout: int = GRAPH_TIMEOUT):
    req_headers = dict(headers or globals().get("headers") or {})
    if json_payload is not None:
        req_headers["Content-Type"] = "application/json"

    refreshed = False
    last_exc = None
    for attempt in range(GRAPH_MAX_RETRIES + 1):
        try:
            resp = requests.request(method, url, headers=req_headers, json=json_payload, data=data, timeout=timeout)
        except requests.exceptions.RequestException as exc:
            last_exc = exc
            if attempt < GRAPH_MAX_RETRIES:
                wait = GRAPH_BACKOFF_BASE ** (attempt + 1)
                logger.warning(f"  [NET] {exc}; retrying in {wait:.0f}s")
                time.sleep(wait)
                continue
            raise

        if resp.status_code == 401 and not refreshed and _refresh_graph_headers_silently():
            req_headers = dict(globals().get("headers") or {})
            if json_payload is not None:
                req_headers["Content-Type"] = "application/json"
            refreshed = True
            logger.info("  [401] Retrying with refreshed token")
            continue

        if resp.status_code == 429 and attempt < GRAPH_MAX_RETRIES:
            retry_after_raw = resp.headers.get("Retry-After")
            try:
                wait = min(int(retry_after_raw), 60) if retry_after_raw else min(int(GRAPH_BACKOFF_BASE ** (attempt + 1)), 60)
            except (TypeError, ValueError):
                wait = min(int(GRAPH_BACKOFF_BASE ** (attempt + 1)), 60)
            logger.warning(f"  [429] Rate limited; retrying in {wait}s")
            time.sleep(wait)
            continue

        if resp.status_code >= 500 and attempt < GRAPH_MAX_RETRIES:
            wait = GRAPH_BACKOFF_BASE ** (attempt + 1)
            logger.warning(f"  [5xx] Server error {resp.status_code}; retrying in {wait:.0f}s")
            time.sleep(wait)
            continue

        return resp

    if last_exc is not None:
        raise last_exc
    raise RuntimeError(f"Graph request failed: {method} {url}")


def graph_get(url: str, headers=None, timeout: int = GRAPH_TIMEOUT):
    return _graph_request("GET", url, headers=headers, timeout=timeout)


def graph_post(url: str, headers=None, json_payload=None, timeout: int = GRAPH_TIMEOUT):
    return _graph_request("POST", url, headers=headers, json_payload=json_payload, timeout=timeout)


print("🚀 Opening browser for authentication...")
interactive_result = app.acquire_token_interactive(scopes=SCOPES, prompt="select_account")

if "access_token" not in interactive_result:
    raise RuntimeError(f"Login Failed: {interactive_result.get('error_description')}")

headers = {"Authorization": f"Bearer {interactive_result['access_token']}"}
logger.info("✅ Login Successful")
logger.info(f"Log File: {log_file}")

In [ ]:
# === Cell 2: SharePoint API & App Selector (v3.6 with Counts & Filtering) ===
import ipywidgets as widgets
from IPython.display import display, clear_output
import requests


def _graph_paginated_values(url: str) -> list[dict]:
    items = []
    next_url = url
    while next_url:
        resp = graph_get(next_url, headers=headers)
        resp.raise_for_status()
        payload = resp.json()
        items.extend(payload.get("value", []))
        next_url = payload.get("@odata.nextLink")
    return items


# --- 1. SharePoint API Functions ---
def get_site_id(site_name: str) -> str:
    url = f"https://graph.microsoft.com/v1.0/sites/{SHAREPOINT_HOST}:/sites/{site_name}"
    resp = graph_get(url, headers=headers)
    resp.raise_for_status()
    return resp.json()["id"]


def list_folders_with_count(site_id: str, path: str) -> list[dict]:
    if not path or path.strip() == "":
        url = f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive/root/children?$select=name,webUrl,folder"
    else:
        clean_path = path.strip("/")
        url = f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive/root:/{clean_path}:/children?$select=name,webUrl,folder"

    results = []
    excluded_names = ["forms", "_private", "audit logs", "audit", "user listings", "shared documents"]
    for item in _graph_paginated_values(url):
        if item.get("folder"):
            name_lower = item["name"].lower()
            if any(ex in name_lower for ex in excluded_names):
                continue
            results.append({
                "name": item["name"],
                "webUrl": item.get("webUrl", ""),
                "count": item.get("folder", {}).get("childCount", 0),
            })
    return results


def list_excel_files(site_id: str, folder_path: str) -> list[dict]:
    clean_path = folder_path.strip("/")
    url = (
        f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive/root:/{clean_path}:/children"
        "?$select=id,name,lastModifiedDateTime,createdDateTime,webUrl,file"
    )
    files = []
    for item in _graph_paginated_values(url):
        if str(item.get("name", "")).lower().endswith(".xlsx"):
            files.append({
                "id": item["id"], "name": item["name"],
                "lastModifiedDateTime": item.get("lastModifiedDateTime"),
                "createdDateTime": item.get("createdDateTime"),
                "webUrl": item.get("webUrl"),
            })
    return sorted(files, key=lambda f: f.get("lastModifiedDateTime", ""), reverse=True)


def download_file(site_id: str, file_path: str) -> bytes:
    clean_path = file_path.strip("/")
    url = f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive/root:/{clean_path}:/content"
    resp = graph_get(url, headers=headers)
    resp.raise_for_status()
    return resp.content


def get_file_audit_info(site_id: str, file_id: str) -> dict:
    url = (
        f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive/items/{file_id}/versions"
        "?$select=lastModifiedDateTime,lastModifiedBy"
    )
    default_res = {"log": "N/A", "creator": "Unknown", "modifier": "Unknown", "created_ts": None}
    try:
        versions = _graph_paginated_values(url)
    except Exception:
        return default_res
    if not versions:
        return default_res

    logs = []
    for v in versions:
        mod_time = v.get("lastModifiedDateTime", "")[:19].replace("T", " ")
        actor = v.get("lastModifiedBy", {}).get("user", {}).get("displayName") or "System"
        logs.append(f"{mod_time} - {actor}")

    last_v = versions[0]
    first_v = versions[-1]
    return {
        "log": "\n".join(logs),
        "creator": first_v.get("lastModifiedBy", {}).get("user", {}).get("displayName") or "System",
        "modifier": last_v.get("lastModifiedBy", {}).get("user", {}).get("displayName") or "System",
        "created_ts": first_v.get("lastModifiedDateTime"),
    }


# --- 2. Initialize Connection ---
try:
    site_id = get_site_id(SITE_NAME)
    logger.info(f"✅ SharePoint Connected (Site ID: {site_id[:10]}...)")
except Exception as e:
    logger.error(f"❌ Connection Failed: {e}")


# --- 3. Tree Selector UI (with Counts) ---
TARGET_APPS = []
USE_CACHE = True


def create_app_selector():
    print(f"📂 Reading Root: {BASE_PATH} ...")
    try:
        categories = list_folders_with_count(site_id, BASE_PATH)
    except Exception as e:
        print(f"❌ Read Failed: {e}")
        return

    ui_container = widgets.VBox()
    app_checkboxes = []
    app_checkbox_map = {}

    chk_cache = widgets.Checkbox(value=True, description="⚡ Use Cache (Faster)", indent=False)
    ui_container.children += (widgets.HTML("<h3>📂 App Selector (v3.6)</h3>"), chk_cache, widgets.HTML("<hr>"))

    def _build_folder_select_all_helpers(select_all_toggle, folder_checkboxes, folder_state):
        def sync_select_all():
            folder_state["syncing"] = True
            if folder_checkboxes:
                select_all_toggle.value = all(cb.value for cb in folder_checkboxes)
            else:
                select_all_toggle.value = False
            folder_state["syncing"] = False

        def on_select_all(change):
            if change.get("name") != "value" or folder_state["syncing"]:
                return
            for cb in folder_checkboxes:
                cb.value = change["new"]

        return sync_select_all, on_select_all

    for cat in categories:
        if cat['name'] in ["Forms", "_private"] or "audit" in cat['name'].lower():
            continue

        cat_label = widgets.HTML(f"<b>📁 {cat['name']}</b>", layout=widgets.Layout(width='150px'))
        btn_expand = widgets.Button(description="➕ Expand", button_style='info', layout=widgets.Layout(width='80px'))
        pbar = widgets.IntProgress(value=0, min=0, max=1, layout=widgets.Layout(width='160px', visibility='hidden'))
        app_list_box = widgets.VBox(layout=widgets.Layout(margin='0 0 0 30px', display='none'))
        select_all_toggle = widgets.Checkbox(value=False, description="Select All", indent=False, layout=widgets.Layout(width='140px'))
        select_all_hint = widgets.HTML("<span style='color:#666; font-size:11px'>Apply to all apps in this folder</span>")
        select_all_row = widgets.HBox(
            [select_all_toggle, select_all_hint],
            layout=widgets.Layout(display='none', margin='0 0 4px 0')
        )
        folder_checkboxes = []
        folder_state = {"syncing": False}
        sync_select_all, on_select_all = _build_folder_select_all_helpers(select_all_toggle, folder_checkboxes, folder_state)
        select_all_toggle.observe(on_select_all, names="value")

        def on_expand(
            b,
            cat_name=cat['name'],
            container=app_list_box,
            btn=btn_expand,
            pbar=pbar,
            select_all_row=select_all_row,
            folder_checkboxes=folder_checkboxes,
            sync_select_all=sync_select_all,
        ):
            if btn.description == "➕ Expand":
                btn.description = "⏳"
                pbar.layout.visibility = 'visible'
                pbar.bar_style = 'info'
                pbar.value = 0
                pbar.max = 1
                sub_path = f"{BASE_PATH}/{cat_name}"
                try:
                    apps = list_folders_with_count(site_id, sub_path)
                    apps = [a for a in apps if a.get('name') not in ["Forms", "_private"]]
                    checks = []
                    folder_checkboxes.clear()
                    if not apps:
                        select_all_row.layout.display = 'none'
                        checks.append(widgets.Label("(Empty)"))
                        pbar.value = 1
                    else:
                        select_all_row.layout.display = 'flex'
                        pbar.max = len(apps)
                        for idx, app_item in enumerate(apps, 1):
                            count_value = int(app_item.get('count', 0))
                            count_color = "#c62828" if count_value < 5 else "#888"
                            count_display = f" <span style='color:{count_color}; font-size:11px'>({count_value} users)</span>"
                            app_key = f"{cat_name}|{app_item['name']}"
                            cb = app_checkbox_map.get(app_key)
                            if cb is None:
                                cb = widgets.Checkbox(value=False, description=app_item['name'], indent=False, layout=widgets.Layout(width='400px'))
                                app_checkbox_map[app_key] = cb
                                app_checkboxes.append(cb)
                            bound_categories = set(getattr(cb, "_select_all_bound_categories", set()))
                            if cat_name not in bound_categories:
                                def _on_child_change(change, sync_select_all=sync_select_all):
                                    if change.get("name") == "value":
                                        sync_select_all()
                                cb.observe(_on_child_change, names="value")
                                bound_categories.add(cat_name)
                                cb._select_all_bound_categories = bound_categories
                            cb.description = app_item['name']
                            cb.app_data = (cat_name, app_item['name'], f"{sub_path}/{app_item['name']}")
                            folder_checkboxes.append(cb)
                            lbl_count = widgets.HTML(count_display)
                            checks.append(widgets.HBox([cb, lbl_count]))
                            pbar.value = idx
                        sync_select_all()
                    container.children = tuple(([select_all_row] if folder_checkboxes else []) + checks)
                    container.layout.display = 'block'
                    btn.description = "➖ Collapse"
                except Exception as e:
                    container.children = (widgets.Label(f"Error: {e}"),)
                    btn.description = "❌"
                finally:
                    pbar.layout.visibility = 'hidden'
                    pbar.bar_style = ''
            else:
                if container.layout.display == 'none':
                    container.layout.display = 'block'; btn.description = "➖ Collapse"
                else:
                    container.layout.display = 'none'; btn.description = "➕ Expand"

        btn_expand.on_click(on_expand)
        ui_container.children += (widgets.HBox([btn_expand, cat_label, pbar]), app_list_box)

    btn_confirm = widgets.Button(description="✅ Confirm Selection", button_style='success', layout=widgets.Layout(margin='20px 0'))
    output_area = widgets.Output()

    def on_confirm(b):
        global TARGET_APPS, USE_CACHE
        TARGET_APPS = [cb.app_data for cb in app_checkboxes if cb.value]
        USE_CACHE = chk_cache.value

        with output_area:
            clear_output()
            if not TARGET_APPS:
                print("⚠️ No apps selected!")
            else:
                print(f"🎯 Selected {len(TARGET_APPS)} Apps | Cache: {USE_CACHE}")
                print("⏳ Proceed to Cell 4 (Enrichment) or Cell 5 (Scan).")

    btn_confirm.on_click(on_confirm)
    display(ui_container, btn_confirm, output_area)


create_app_selector()


In [ ]:
# === Cell 5: AER Engine v4.2 (Cache Fix, User Info, Global Report, Safe Write) ===

import pandas as pd

import re, time, json, os, requests

import ipywidgets as widgets

from datetime import datetime

from openpyxl import load_workbook
from openpyxl.styles import Alignment
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

from io import BytesIO

from IPython.display import display, HTML, clear_output

import importlib.util
from IPython import get_ipython

def _widgets_env_ready():
    try:
        ip = get_ipython()
        if ip is None or ip.__class__.__name__ != 'ZMQInteractiveShell':
            return False
        has_nb = importlib.util.find_spec('widgetsnbextension') is not None
        has_lab = importlib.util.find_spec('jupyterlab_widgets') is not None
        return has_nb or has_lab
    except Exception:
        return False

USE_WIDGETS = _widgets_env_ready()


# ==========================================

# PART 1: CONFIG & LOADER

# ==========================================

CACHE_FILE = "aer_cache.json"

NOTES_FILE = "aer_manual_notes.json"

CACHE_VERSION = 4.2

cache_updated = False

EXCLUDED_FOLDERS = ["forms", "_private", "user listings", "audit logs", "audit"]



def load_json(file_path):

    if os.path.exists(file_path):

        try:

            with open(file_path, 'r', encoding='utf-8') as f: return json.load(f)

        except: return {}

    return {}



def save_json(file_path, data):

    directory = os.path.dirname(os.path.abspath(file_path))

    os.makedirs(directory, exist_ok=True)

    tmp_path = f"{file_path}.tmp"

    with open(tmp_path, 'w', encoding='utf-8') as f:

        json.dump(data, f, indent=2, ensure_ascii=False)

    os.replace(tmp_path, file_path)



def safe_excel_path(base_path):

    """If base_path is locked/open, append _1, _2, ... until writable."""

    if not os.path.exists(base_path):

        return base_path

    try:

        with open(base_path, 'a'):

            pass

        return base_path

    except OSError:

        pass

    stem, ext = os.path.splitext(base_path)

    for i in range(1, 100):

        candidate = f"{stem}_{i}{ext}"

        if not os.path.exists(candidate):

            return candidate

    return f"{stem}_{int(time.time())}{ext}"



def format_export_excel(file_path, audit_col_name="Audit Log"):

    wb = load_workbook(file_path)

    ws = wb.active

    header_to_col = {}

    for col_idx in range(1, ws.max_column + 1):

        hdr = ws.cell(row=1, column=col_idx).value

        hdr_txt = str(hdr).strip() if hdr is not None else ""

        if hdr_txt:

            header_to_col[hdr_txt] = col_idx

        max_len = len(hdr_txt)

        for row_idx in range(2, ws.max_row + 1):

            cell_val = ws.cell(row=row_idx, column=col_idx).value

            if cell_val is None:

                continue

            lines = str(cell_val).splitlines() or [str(cell_val)]

            max_len = max(max_len, max(len(line) for line in lines))

        ws.column_dimensions[get_column_letter(col_idx)].width = min(max(10, max_len + 2), 80)

    audit_col = header_to_col.get(audit_col_name)

    if audit_col:

        for row_idx in range(2, ws.max_row + 1):

            cell = ws.cell(row=row_idx, column=audit_col)

            txt = "" if cell.value is None else str(cell.value)

            line_count = max(1, txt.count("\n") + 1)

            cell.alignment = Alignment(wrap_text=True, vertical="top")

            current_height = ws.row_dimensions[row_idx].height or 15

            ws.row_dimensions[row_idx].height = max(current_height, min(15 * line_count, 300))

    wb.save(file_path)


def _find_header_column(ws, header_row, header_name):

    target = str(header_name).strip()

    for col_idx in range(1, ws.max_column + 1):

        cell_value = ws.cell(row=header_row, column=col_idx).value

        if cell_value is None:

            continue

        if str(cell_value).strip() == target:

            return col_idx

    return None


def _set_header_widths(ws, header_row, width_map):

    for header_name, width in width_map.items():

        col_idx = _find_header_column(ws, header_row, header_name)

        if col_idx is None:

            continue

        column_letter = get_column_letter(col_idx)

        current_width = ws.column_dimensions[column_letter].width

        if current_width is None or float(current_width) == 13.0:

            ws.column_dimensions[column_letter].width = width

        else:

            ws.column_dimensions[column_letter].width = max(current_width, width)


def _add_excel_table(ws, start_row, end_row, start_col, end_col, table_name):

    if end_row <= start_row or end_col < start_col:

        return

    ref = f"{get_column_letter(start_col)}{start_row}:{get_column_letter(end_col)}{end_row}"

    table = Table(displayName=table_name, ref=ref)

    table.tableStyleInfo = TableStyleInfo(
        name="TableStyleMedium2",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=False,
    )

    ws.add_table(table)


def _apply_short_hyperlinks(ws, header_row, header_name="Folder URL", display_text="Open", start_data_row=None):

    col_idx = _find_header_column(ws, header_row, header_name)

    if col_idx is None:

        return

    data_row = start_data_row or (header_row + 1)

    for row_idx in range(data_row, ws.max_row + 1):

        cell = ws.cell(row=row_idx, column=col_idx)

        raw_value = "" if cell.value is None else str(cell.value).strip()

        if not raw_value:

            cell.value = None

            continue

        cell.value = display_text

        cell.hyperlink = raw_value

        cell.style = "Hyperlink"


def format_global_report_excel(file_path, summary_sheet_name="Sheet1", progress_sheet_name="Progress", remaining_header_row=None):

    wb = load_workbook(file_path)

    summary_ws = wb[summary_sheet_name]

    progress_ws = wb[progress_sheet_name]

    _set_header_widths(summary_ws, 1, {
        "Category": 16,
        "App Name": 28,
        "Reviewer": 24,
        "Progress": 18,
        "Total Approved": 14,
        "Total Denied": 14,
        "Total Changed": 14,
    })

    if summary_ws.max_row > 1 and summary_ws.max_column > 0:

        _add_excel_table(
            summary_ws,
            start_row=1,
            end_row=summary_ws.max_row,
            start_col=1,
            end_col=summary_ws.max_column,
            table_name="SummaryTable",
        )

    _set_header_widths(progress_ws, 1, {
        "Metric": 30,
        "Value": 14,
    })

    if remaining_header_row is not None and remaining_header_row <= progress_ws.max_row:

        _set_header_widths(progress_ws, remaining_header_row, {
            "Category": 16,
            "App Name": 28,
            "Reviewer": 24,
            "Missing": 10,
            "Total Rows": 10,
            "Completion %": 12,
            "Folder URL": 10,
        })

        _apply_short_hyperlinks(
            progress_ws,
            header_row=remaining_header_row,
            header_name="Folder URL",
            display_text="Open",
            start_data_row=remaining_header_row + 1,
        )

    wb.save(file_path)



local_cache = load_json(CACHE_FILE)

manual_data_store = load_json(NOTES_FILE)



# ==========================================

# PART 2: ROBUST EXCEL PARSER

# ==========================================

def find_col_index(headers, keywords):

    """Case-insensitive search for column index."""

    for idx, h in enumerate(headers):

        if not h: continue

        h_str = str(h).strip().lower()

        if all(k in h_str for k in keywords):

            return idx

    return None



def is_name_like_header(header_text: str) -> bool:

    txt = (header_text or "").strip().lower()

    if not txt:

        return False

    if "reviewer" in txt or "manager" in txt:

        return False

    return ("name" in txt) or ("display" in txt and "user" in txt) or ("full" in txt and "name" in txt)



def is_email_like_header(header_text: str) -> bool:

    txt = (header_text or "").strip().lower()

    if not txt:

        return False

    return ("email" in txt) or ("mail" == txt) or txt.endswith(" mail")



def resolve_column_map(header, app_col_map=None):

    if app_col_map:

        return app_col_map, "app-locked"

    idx_rev = find_col_index(header, ["reviewer"])

    idx_res = find_col_index(header, ["response"])

    idx_det = find_col_index(header, ["details", "change"])

    idx_user = None

    idx_email = None

    if idx_rev is not None:

        rev_hdr = str(header[idx_rev]).strip().lower() if header[idx_rev] else ""

        if "response" in rev_hdr:

            idx_rev = None

            for i, h in enumerate(header):

                h_str = str(h).strip().lower() if h else ""

                if ("reviewer" in h_str) and ("response" not in h_str):

                    idx_rev = i

                    break

    for i in range(min(2, len(header))):

        h = str(header[i]).strip().lower() if header[i] else ""

        if idx_user is None and is_name_like_header(h):

            idx_user = i

        if idx_email is None and is_email_like_header(h):

            idx_email = i

    if idx_user is None: idx_user = find_col_index(header, ["user", "name"])

    if idx_user is None: idx_user = find_col_index(header, ["display", "name"])

    if idx_user is None: idx_user = find_col_index(header, ["full", "name"])

    if idx_user is None: idx_user = find_col_index(header, ["name"])

    if idx_email is None: idx_email = find_col_index(header, ["email"])

    if idx_email is None: idx_email = find_col_index(header, ["mail"])

    if idx_rev is None: idx_rev = find_col_index(header, ["manager"])

    if idx_rev is None or idx_res is None:

        return None, "invalid"

    return {

        "idx_rev": idx_rev, "idx_res": idx_res, "idx_det": idx_det,

        "idx_user": idx_user, "idx_email": idx_email

    }, "detected"



def read_excel_rows(excel_bytes: bytes, reviewer_name: str, file_name: str, folder_url: str, app_col_map=None):

    wb = load_workbook(BytesIO(excel_bytes), read_only=True)



    # 1. Smart Tab Selection

    sheet_name = wb.sheetnames[0]

    for sn in wb.sheetnames:

        if "user listing" in sn.lower(): sheet_name = sn; break

    ws = wb[sheet_name]



    # 2. Read Headers

    rows_iter = ws.iter_rows(values_only=True)

    try:

        header = next(rows_iter)

    except StopIteration:

        return [], None, "empty-sheet"



    # 3. Column Mapping

    col_map, map_source = resolve_column_map(header, app_col_map=app_col_map)

    if not col_map:

        return [], None, map_source

    idx_rev = col_map["idx_rev"]

    idx_res = col_map["idx_res"]

    idx_det = col_map["idx_det"]

    idx_user = col_map["idx_user"]

    idx_email = col_map["idx_email"]



    results = []



    # 4. Iterate Rows

    for i, row in enumerate(rows_iter, start=2):

        r_rev = row[idx_rev] if idx_rev < len(row) else None

        r_res = row[idx_res] if idx_res < len(row) else None

        r_det = row[idx_det] if idx_det is not None and idx_det < len(row) else None

        r_user = row[idx_user] if idx_user is not None and idx_user < len(row) else None

        r_email = row[idx_email] if idx_email is not None and idx_email < len(row) else None



        if str(r_rev).strip().lower() != reviewer_name.lower(): continue



        val_res = str(r_res).strip() if r_res else ""

        val_det = str(r_det).strip() if r_det else ""



        results.append({

            "reviewer": reviewer_name,

            "user_name": str(r_user).strip() if r_user else "",

            "user_email": str(r_email).strip() if r_email else "",

            "response": val_res,

            "details": val_det,

            "is_missing": (val_res == ""),

            "row_number": i,

            "file_name": file_name,

            "folder_url": folder_url

        })

    return results, col_map, map_source



def get_row_stats(txt):

    txt = str(txt).lower().strip()

    kw_appr = ['approv', 'retain', 'keep', 'confirm', 'yes', 'ok', 'active']

    kw_deny = ['denied', 'deny', 'remove', 'delete', 'revok', 'reject', 'no']

    kw_chg  = ['change', 'modif', 'updat', 'correct', 'edit', 'adjust']

    return {

        "is_appr": 1 if any(k in txt for k in kw_appr) else 0,

        "is_deny": 1 if any(k in txt for k in kw_deny) else 0,

        "is_chg":  1 if any(k in txt for k in kw_chg) else 0

    }



# ==========================================

# PART 3: SCANNING ENGINE

# ==========================================

if 'TARGET_APPS' not in globals() or not TARGET_APPS:

    print("⚠️ Please select Apps in Cell 2 first!")

    TARGET_APPS = []



all_responses = []

errors = []

app_column_map_store = {}



print(f"🚀 Starting Scan Engine v{CACHE_VERSION} (Fuzzy Column Match)...")



# UI: Per-app scan progress (details stay in log file)
scan_progress_box = widgets.VBox()
scan_progress_store = {}

if TARGET_APPS:
    scan_rows = []
    for category, app_name, app_path in TARGET_APPS:
        ui_key = f"{category}|{app_name}|{app_path}"
        w_label = widgets.HTML(
            f"<b>{category} &gt; {app_name}</b>",
            layout=widgets.Layout(width="360px"),
        )
        w_bar = widgets.IntProgress(
            value=0,
            min=0,
            max=1,
            bar_style="",
            layout=widgets.Layout(width="260px"),
        )
        w_status = widgets.HTML("Queued", layout=widgets.Layout(width="120px"))
        scan_progress_store[ui_key] = {"bar": w_bar, "status": w_status}
        scan_rows.append(
            widgets.HBox([w_label, w_bar, w_status], layout=widgets.Layout(align_items="center"))
        )
    scan_progress_box.children = tuple(scan_rows)
    display(widgets.VBox([widgets.HTML("<h4>📡 Scan Progress (Per App)</h4>"), scan_progress_box]))


for category, current_app_name, current_path in TARGET_APPS:

    ui_key = f"{category}|{current_app_name}|{current_path}"
    app_ui = scan_progress_store.get(ui_key)

    try:

        if app_ui:
            app_ui["bar"].bar_style = "info"
            app_ui["bar"].value = 0
            app_ui["bar"].max = 1
            app_ui["status"].value = "Loading..."

        raw_folders = list_folders_with_count(site_id, current_path) if 'list_folders_with_count' in globals() else []

        reviewers = [r for r in raw_folders if r['name'].lower() not in EXCLUDED_FOLDERS]

        total_revs = len(reviewers)

        if app_ui:
            app_ui["bar"].max = max(1, total_revs)
            app_ui["bar"].value = 0
            app_ui["status"].value = f"0/{total_revs}"

        logger.info(f"📂 App: {current_app_name} | Reviewers: {total_revs}")

        app_schema_key = f"{category}|{current_app_name}"

        app_col_map = app_column_map_store.get(app_schema_key)



        from concurrent.futures import ThreadPoolExecutor, as_completed

        def _scan_one(folder, idx, app_col_map):
            reviewer_name = folder["name"]
            folder_url = folder["webUrl"]
            folder_path = f"{current_path}/{reviewer_name}"
            cache_key = f"{category}|{current_app_name}|{reviewer_name}"

            try:
                files = list_excel_files(site_id, folder_path)
                match_candidates = [f for f in files if reviewer_name.lower() in f["name"].lower()]
                target_file = match_candidates[0] if match_candidates else (files[0] if files else None)

                if not target_file:
                    logger.info(f"  ⚠️ Skip: [{idx}/{total_revs}] {reviewer_name} (no xlsx found)")
                    return {"skip": True, "idx": idx, "reviewer": reviewer_name}

                remote_mod = target_file.get("lastModifiedDateTime")
                logger.info(f"  🔎 File: [{idx}/{total_revs}] {reviewer_name} | Target:{target_file['name']} | Modified:{remote_mod}")

                cached = local_cache.get(cache_key)
                is_hit = False
                force_live_recheck = False
                cache_pending = False

                if USE_CACHE and cached and cached.get('v') == CACHE_VERSION and cached.get('last_mod') == remote_mod and 'rows' in cached and len(cached['rows']) > 0:
                    cache_pending = any(r.get('is_missing') for r in cached.get('rows', []))
                    if cache_pending:
                        force_live_recheck = True
                        logger.info(f"  ♻️ Cache Recheck: [{idx}/{total_revs}] {reviewer_name} has pending rows; reading live and overwriting cache.")
                    else:
                        is_hit = True
                        logger.info(f"  🧊 Cache Hit (Completed): [{idx}/{total_revs}] {reviewer_name} | rows:{len(cached['rows'])}")
                else:
                    if not USE_CACHE:
                        logger.info(f"  ℹ️ Cache Bypass: [{idx}/{total_revs}] {reviewer_name} (USE_CACHE=False)")
                    elif not cached:
                        logger.info(f"  ℹ️ Cache Miss: [{idx}/{total_revs}] {reviewer_name} (no cache key)")
                    elif cached.get('v') != CACHE_VERSION:
                        logger.info(f"  ℹ️ Cache Miss: [{idx}/{total_revs}] {reviewer_name} (version {cached.get('v')} != {CACHE_VERSION})")
                    elif cached.get('last_mod') != remote_mod:
                        logger.info(f"  ℹ️ Cache Miss: [{idx}/{total_revs}] {reviewer_name} (modified changed)")
                    else:
                        logger.info(f"  ℹ️ Cache Miss: [{idx}/{total_revs}] {reviewer_name} (no cached rows)")

                if is_hit:
                    audit_snap = cached.get('audit', {})
                    c_appr = cached['stats']['Appr']
                    c_deny = cached['stats']['Deny']
                    c_chg = cached['stats']['Chg']
                    c_miss = sum(1 for r in cached['rows'] if r['is_missing'])

                    logger.info(f"  ✅ Read (Cache): [{idx}/{total_revs}] {reviewer_name} (Missing:{c_miss})(A:{c_appr}, D:{c_deny}, C:{c_chg})")

                    rows_out = []
                    for r in cached['rows']:
                        r_copy = r.copy()
                        st = get_row_stats(r['response'])
                        r_copy.update({
                            "Category": category, "App_Name": current_app_name,
                            "Last_Modified": remote_mod, "File_Created_Date": audit_snap.get('created_ts'),
                            "Audit_Log": audit_snap.get('log'), "File_Creator": audit_snap.get('creator'), "File_Modifier": audit_snap.get('modifier'),
                            "stats_appr": st['is_appr'], "stats_deny": st['is_deny'], "stats_chg": st['is_chg'],
                            "source_is_cache": True
                        })
                        rows_out.append(r_copy)

                    return {
                        "rows": rows_out,
                        "cache_entry": None,
                        "cache_key": cache_key,
                        "detected_col_map": None,
                        "map_source": None,
                        "idx": idx,
                        "total": total_revs,
                        "reviewer": reviewer_name,
                        "folder_url": folder_url,
                        "stats": {"Appr": c_appr, "Deny": c_deny, "Chg": c_chg, "Miss": c_miss},
                        "force_live_recheck": False,
                        "read_mode": "cache"
                    }

                content = download_file(site_id, f"{folder_path}/{target_file['name']}")

                rows, detected_col_map, map_source = read_excel_rows(
                    content, reviewer_name, target_file['name'], folder_url, app_col_map=app_col_map
                )

                audit_info = {
                    "log": "N/A",
                    "creator": "Unknown",
                    "modifier": "Unknown",
                    "created_ts": target_file.get("createdDateTime"),
                }
                if rows:
                    audit_info = get_file_audit_info(site_id, target_file["id"])

                s_appr, s_deny, s_chg, miss_cnt = 0, 0, 0, 0
                clean_rows_cache = []
                final_created = audit_info.get('created_ts') or target_file.get("createdDateTime")

                for r in rows:
                    st = get_row_stats(r['response'])
                    s_appr += st['is_appr']; s_deny += st['is_deny']; s_chg += st['is_chg']
                    if r['is_missing']: miss_cnt += 1

                    clean_rows_cache.append({
                        "reviewer": r['reviewer'], "user_name": r.get('user_name', ''), "user_email": r.get('user_email', ''),
                        "response": r['response'], "details": r['details'],
                        "is_missing": r['is_missing'], "row_number": r['row_number'],
                        "file_name": r['file_name'], "folder_url": r['folder_url']
                    })

                    r.update({
                        "Category": category, "App_Name": current_app_name,
                        "Last_Modified": remote_mod, "File_Created_Date": final_created,
                        "Audit_Log": audit_info['log'], "File_Creator": audit_info['creator'], "File_Modifier": audit_info['modifier'],
                        "stats_appr": st['is_appr'], "stats_deny": st['is_deny'], "stats_chg": st['is_chg'],
                        "source_is_cache": False
                    })

                logger.info(
                    f"  ✅ Read (Live): [{idx}/{total_revs}] {reviewer_name} "
                    f"(Rows:{len(rows)} Missing:{miss_cnt})(A:{s_appr}, D:{s_deny}, C:{s_chg})"
                )

                cache_entry = None
                if rows or force_live_recheck:
                    cache_entry = {
                        "v": CACHE_VERSION, "last_mod": remote_mod,
                        "stats": {"Appr": s_appr, "Deny": s_deny, "Chg": s_chg},
                        "audit": audit_info, "rows": clean_rows_cache
                    }

                return {
                    "rows": rows,
                    "cache_entry": cache_entry,
                    "cache_key": cache_key,
                    "detected_col_map": detected_col_map,
                    "map_source": map_source,
                    "idx": idx,
                    "total": total_revs,
                    "reviewer": reviewer_name,
                    "folder_url": folder_url,
                    "stats": {"Appr": s_appr, "Deny": s_deny, "Chg": s_chg, "Miss": miss_cnt,
                              "CacheRows": len(clean_rows_cache), "ForceRecheck": force_live_recheck},
                    "force_live_recheck": force_live_recheck,
                    "read_mode": "live"
                }
            except Exception as e:
                return {"error": str(e), "reviewer": reviewer_name, "folder_url": folder_url, "idx": idx, "total": total_revs}

        done_count = 0
        max_workers = 4
        futures = []
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            for idx, folder in enumerate(reviewers, 1):
                futures.append(executor.submit(_scan_one, folder, idx, app_col_map))

            for fut in as_completed(futures):
                result = fut.result()
                if result.get("skip"):
                    done_count += 1
                elif result.get("error"):
                    logger.error(f"  ❌ Error {result.get('reviewer', '')}: {result['error']}")
                    errors.append({"Category": category, "App_Name": current_app_name, "reviewer": result.get("reviewer", ""), "error": result["error"], "folder_url": result.get("folder_url", "#")})
                    done_count += 1
                else:
                    if result.get("read_mode") == "live":
                        detected_col_map = result.get("detected_col_map")
                        map_source = result.get("map_source")
                        if detected_col_map and not app_col_map:
                            app_col_map = detected_col_map
                            app_column_map_store[app_schema_key] = detected_col_map
                            logger.info(
                                f"  🧭 App Column Map Locked: NameIdx={detected_col_map.get('idx_user')} "
                                f"EmailIdx={detected_col_map.get('idx_email')} Source={map_source}"
                            )
                        elif detected_col_map and map_source == "app-locked":
                            logger.info(
                                f"  🧭 App Column Map Reused: NameIdx={detected_col_map.get('idx_user')} "
                                f"EmailIdx={detected_col_map.get('idx_email')}"
                            )
                        elif not detected_col_map:
                            logger.warning(f"  ⚠️ Column Mapping Failed: [{result['idx']}/{result['total']}] {result['reviewer']}")

                    if result.get("rows"):
                        all_responses.extend(result["rows"])

                    cache_entry = result.get("cache_entry")
                    if cache_entry is not None:
                        local_cache[result["cache_key"]] = cache_entry
                        cache_updated = True
                        stats = result.get("stats", {})
                        logger.info(
                            f"  💾 Cache Write: [{result['idx']}/{result['total']}] {result['reviewer']} "
                            f"(Rows:{stats.get('CacheRows', 0)} Missing:{stats.get('Miss', 0)} PendingRecheck:{stats.get('ForceRecheck', False)})"
                        )

                    done_count += 1

                if app_ui:
                    app_ui["bar"].value = min(done_count, app_ui["bar"].max)
                    app_ui["status"].value = f"{done_count}/{total_revs}"
        if app_ui:
            app_ui["bar"].bar_style = "success"
            if total_revs <= 0:
                app_ui["bar"].value = 1
                app_ui["bar"].max = 1
                app_ui["status"].value = "0/0 Done"
            else:
                app_ui["bar"].value = total_revs
                app_ui["status"].value = f"{total_revs}/{total_revs} Done"

    except Exception as e:

        logger.error(f"❌ App Error: {e}")

        if app_ui:

            app_ui["bar"].bar_style = "danger"

            app_ui["status"].value = "Error"



if cache_updated:

    save_json(CACHE_FILE, local_cache)

    logger.info("💾 Cache Updated (v4.2)")



# ==========================================

# PART 4: DASHBOARD

# ==========================================

df = pd.DataFrame(all_responses)

widget_store = {}

unified_data = {}

today_str = datetime.now().strftime("%Y-%m-%d")

report_dir = f"output/{today_str}/report"

os.makedirs(report_dir, exist_ok=True)



if not df.empty:

    stats = df.groupby(["Category", "App_Name", "reviewer"]).agg(

        total_rows=("reviewer", "size"),

        missing=("is_missing", "sum"),

        approved=("stats_appr", "sum"), denied=("stats_deny", "sum"), changed=("stats_chg", "sum"),

        f_creator=("File_Creator", "first"), f_modifier=("File_Modifier", "first"), audit=("Audit_Log", "first"),

        is_cached=("source_is_cache", "all")

    ).reset_index()

    folder_lookup = df.groupby(["Category", "App_Name", "reviewer"]).agg(

        folder_url=("folder_url", "first"),

    ).reset_index()

    folder_lookup_map = {

        (row['Category'], row['App_Name'], row['reviewer']): row.get('folder_url', '#')

        for _, row in folder_lookup.iterrows()

    }



    for _, row in stats.iterrows():

        key = f"{row['Category']} > {row['App_Name']}"

        if key not in unified_data:

            saved_app = manual_data_store.get(key, {})

            unified_data[key] = {

                "Category": row['Category'], "App_Name": row['App_Name'],

                "status_manual": saved_app.get("app_status", "Calculated"), "note_manual": saved_app.get("app_note", ""),

                "reviewers": {}, "stats": {"total_users": 0, "completed_users": 0}

            }



        node = unified_data[key]

        node['stats']['total_users'] += 1

        is_done = (row['missing'] == 0)

        if is_done: node['stats']['completed_users'] += 1

        status_calc = f"❌ Pending: {row['missing']}"

        if is_done: status_calc = "✅ Cached - Completed" if row['is_cached'] else "✅ Completed"



        d_style = "color:red;font-weight:bold" if row['denied'] > 0 else "color:#555"

        node['reviewers'][row['reviewer']] = {

            "status_calc": status_calc,

            "detail_html": f"Appr:{int(row['approved'])} | <span style='{d_style}'>Deny:{int(row['denied'])}</span> | Chg:{int(row['changed'])}",

            "folder_url": folder_lookup_map.get((row['Category'], row['App_Name'], row['reviewer']), '#') or '#',

        }



def _compute_overall_progress(stats_df, raw_df):

    folder_map = raw_df.groupby(["Category", "App_Name", "reviewer"]).agg(
        folder_url=("folder_url", "first"),
    ).reset_index()

    merged = stats_df.merge(folder_map, on=["Category", "App_Name", "reviewer"], how="left")

    total_app_reviewers = int(len(merged))
    pending_app_reviewers = int((merged["missing"] > 0).sum()) if total_app_reviewers else 0
    done_app_reviewers = total_app_reviewers - pending_app_reviewers
    progress_pct = round(done_app_reviewers / total_app_reviewers * 100, 1) if total_app_reviewers else 0.0
    distinct_reviewers_to_notify = int(merged.loc[merged["missing"] > 0, "reviewer"].nunique()) if total_app_reviewers else 0
    distinct_apps_remaining = int(merged.loc[merged["missing"] > 0, "App_Name"].nunique()) if total_app_reviewers else 0

    remaining = merged[merged["missing"] > 0].copy()
    if not remaining.empty:
        denom = remaining["total_rows"].replace(0, pd.NA)
        remaining["Completion %"] = (((remaining["total_rows"] - remaining["missing"]) / denom) * 100).round(1).fillna(0.0)
        remaining = remaining.rename(columns={
            "App_Name": "App Name",
            "reviewer": "Reviewer",
            "missing": "Missing",
            "total_rows": "Total Rows",
            "folder_url": "Folder URL",
        })
        remaining = remaining[["Category", "App Name", "Reviewer", "Missing", "Total Rows", "Completion %", "Folder URL"]]
        remaining = remaining.sort_values(["Missing", "Category", "App Name", "Reviewer"], ascending=[False, True, True, True])
    else:
        remaining = pd.DataFrame(columns=["Category", "App Name", "Reviewer", "Missing", "Total Rows", "Completion %", "Folder URL"])

    overview_df = pd.DataFrame([
        {"Metric": "Total app-reviewers", "Value": total_app_reviewers},
        {"Metric": "Pending app-reviewers", "Value": pending_app_reviewers},
        {"Metric": "Done app-reviewers", "Value": done_app_reviewers},
        {"Metric": "Overall progress (%)", "Value": progress_pct},
        {"Metric": "Distinct reviewers to notify", "Value": distinct_reviewers_to_notify},
        {"Metric": "Distinct app to be completed", "Value": distinct_apps_remaining},
    ])

    return {
        "total_app_reviewers": total_app_reviewers,
        "pending_app_reviewers": pending_app_reviewers,
        "done_app_reviewers": done_app_reviewers,
        "progress_pct": progress_pct,
        "distinct_reviewers_to_notify": distinct_reviewers_to_notify,
        "distinct_apps_remaining": distinct_apps_remaining,
        "overview_df": overview_df,
        "remaining_work_df": remaining,
    }


def _build_remaining_work_table_html(rows, headers, total_row, title):

    if not rows:
        return f"<div style='width:96%; margin:0 auto 8px auto'><b>{title}</b><br><i>✅ No pending app-reviewers. All done.</i></div>"

    header_style = "padding:6px; vertical-align:middle; text-align:left"
    cell_style = "padding:6px; overflow-wrap:anywhere; word-break:break-word; white-space:normal; vertical-align:middle"
    table = (
        "<div style='width:96%; margin:0 auto 10px auto'>"
        f"<div style='font-weight:bold; margin:0 0 4px 0'>{title}</div>"
        "<table border='1' style='width:100%; border-collapse:collapse; font-size:12px; border:1px solid #d8dde6; table-layout:fixed'>"
    )
    table += "<tr style='background:#eef3f8; font-weight:bold'>"
    for header in headers:
        table += f"<th style='{header_style}'>{header}</th>"
    table += "</tr>"
    table += "<tr style='background:#f7f9fc; font-weight:bold'>"
    for header in headers:
        table += f"<td style='{cell_style}'>{total_row.get(header, '')}</td>"
    table += "</tr>"
    for row in rows:
        table += "<tr>"
        for header in headers:
            value = row.get(header, "")
            table += f"<td style='{cell_style}'>{value}</td>"
        table += "</tr>"
    table += "</table></div>"
    return table


def _build_remaining_work_summary_tables(remaining_df):

    if remaining_df is None or remaining_df.empty:
        empty_html = "<i>✅ No pending app-reviewers. All done.</i>"
        return {
            "app_count": 0,
            "reviewer_count": 0,
            "app_html": empty_html,
            "reviewer_html": empty_html,
        }

    app_summary = (
        remaining_df.groupby(["Category", "App Name"], dropna=False)["Reviewer"]
        .nunique()
        .reset_index(name="Remaining Reviewers")
        .sort_values(["Remaining Reviewers", "Category", "App Name"], ascending=[False, True, True], kind="stable")
    )
    reviewer_summary = (
        remaining_df.groupby(["Reviewer"], dropna=False)
        .size()
        .reset_index(name="Remaining Apps")
        .sort_values(["Remaining Apps", "Reviewer"], ascending=[False, True], kind="stable")
    )

    app_rows = app_summary[["Category", "App Name", "Remaining Reviewers"]].to_dict("records")
    reviewer_rows = reviewer_summary[["Reviewer", "Remaining Apps"]].to_dict("records")

    app_headers = ["Category", "App Name", "Remaining Reviewers"]
    reviewer_headers = ["Reviewer", "Remaining Apps"]
    app_total_row = {
        "Category": "Total",
        "App Name": f"{int(len(app_summary))} apps",
        "Remaining Reviewers": int(app_summary["Remaining Reviewers"].sum()),
    }
    reviewer_total_row = {
        "Reviewer": f"Total ({int(len(reviewer_summary))} reviewers)",
        "Remaining Apps": int(reviewer_summary["Remaining Apps"].sum()),
    }

    return {
        "app_count": int(len(app_summary)),
        "reviewer_count": int(len(reviewer_summary)),
        "app_html": _build_remaining_work_table_html(app_rows, app_headers, app_total_row, "By App"),
        "reviewer_html": _build_remaining_work_table_html(reviewer_rows, reviewer_headers, reviewer_total_row, "By Reviewer"),
    }


def _build_remaining_work_html(remaining_df):

    if remaining_df is None or remaining_df.empty:
        return "<i>✅ No pending app-reviewers. All done.</i>"

    cell_style = "padding:6px; overflow-wrap:anywhere; word-break:break-word; white-space:normal; vertical-align:middle"
    table = (
        "<div style='width:100%; overflow:auto'>"
        "<table border='1' style='margin:0 auto; border-collapse:collapse; width:96%; font-size:12px; "
        "border:1px solid #d8dde6; table-layout:fixed'>"
    )
    table += (
        "<tr style='background:#eef3f8; font-weight:bold'>"
        "<th style='padding:6px; width:18%; vertical-align:middle'>Quarter</th>"
        "<th style='padding:6px; width:24%; vertical-align:middle'>App Name</th>"
        "<th style='padding:6px; width:20%; vertical-align:middle'>Reviewer</th>"
        "<th style='padding:6px; width:10%; vertical-align:middle'>Missing</th>"
        "<th style='padding:6px; width:10%; vertical-align:middle'>Total</th>"
        "<th style='padding:6px; width:10%; vertical-align:middle'>Completion %</th>"
        "<th style='padding:6px; width:8%; vertical-align:middle'>Link</th>"
        "</tr>"
    )

    for _, r in remaining_df.iterrows():
        url = r.get("Folder URL") or "#"
        link_html = (
            f"<a href='{url}' target='_blank' rel='noopener noreferrer' "
            "style='background:#0078d4;color:#fff;padding:6px 10px;border-radius:6px;"
            "text-decoration:none;display:inline-block;white-space:nowrap'>Open Folder</a>"
        )
        table += (
            "<tr>"
            f"<td style='{cell_style}'>{r.get('Category', '')}</td>"
            f"<td style='{cell_style}'>{r.get('App Name', '')}</td>"
            f"<td style='{cell_style}'>{r.get('Reviewer', '')}</td>"
            f"<td style='padding:6px; text-align:center; vertical-align:middle'>{int(r.get('Missing', 0))}</td>"
            f"<td style='padding:6px; text-align:center; vertical-align:middle'>{int(r.get('Total Rows', 0))}</td>"
            f"<td style='padding:6px; text-align:center; vertical-align:middle'>{r.get('Completion %', 0)}</td>"
            f"<td style='padding:6px; text-align:center; vertical-align:middle'>{link_html}</td>"
            "</tr>"
        )

    table += "</table></div>"
    return table


def _build_progress_overview_html(total_app_reviewers, pending_app_reviewers, done_app_reviewers, progress_pct, distinct_reviewers_to_notify, distinct_apps_remaining, remaining_df):
    bar_pct = 0.0 if total_app_reviewers == 0 else round(done_app_reviewers / total_app_reviewers * 100, 1)
    metrics_html = (
        f"<b>Total app-reviewers:</b> {total_app_reviewers} &nbsp; "
        f"<b>Pending:</b> {pending_app_reviewers} &nbsp; "
        f"<b>Done:</b> {done_app_reviewers} &nbsp; "
        f"<b>Progress:</b> {progress_pct}% &nbsp; "
        f"<b>Distinct apps remaining:</b> {distinct_apps_remaining} &nbsp; "
        f"<b>Distinct reviewers remaining:</b> {distinct_reviewers_to_notify}"
    )
    bar_html = (
        "<div style='width:420px; height:12px; border:1px solid #cfd7e6; border-radius:6px; overflow:hidden; display:inline-block; vertical-align:middle'>"
        f"<div style='height:12px; width:{bar_pct}%; background:#4caf50'></div>"
        "</div>"
        f"<span style='margin-left:8px; font-size:12px; color:#555'>{done_app_reviewers}/{total_app_reviewers} completed</span>"
    )
    remaining_tables = _build_remaining_work_summary_tables(remaining_df)
    remaining_body = (
        "<div style='display:flex; align-items:center; gap:8px; margin:0 0 6px 2%;'>"
        "<b>Sort by</b>"
        f"<span>App ({remaining_tables['app_count']}) / Reviewer ({remaining_tables['reviewer_count']})</span>"
        "</div>"
        f"{remaining_tables['app_html']}"
        f"{remaining_tables['reviewer_html']}"
    )
    remaining_html = (
        "<details open>"
        "<summary style='font-weight:bold; cursor:pointer'>🧾 Remaining Work (Pending App-Reviewers)</summary>"
        f"{remaining_body}"
        "</details>"
    )
    return (
        "<h4>Progress Overview</h4>"
        f"{metrics_html}<br><br>{bar_html}"
        f"{remaining_html}"
    )

def _build_summary_table_html(stats_df, raw_df):
    if stats_df is None or stats_df.empty:
        return "<i>No app-reviewer data.</i>"
    folder_map = raw_df.groupby(["Category", "App_Name", "reviewer"]).agg(folder_url=("folder_url", "first")).reset_index()
    merged = stats_df.merge(folder_map, on=["Category", "App_Name", "reviewer"], how="left")
    merged["Status"] = merged["missing"].apply(lambda x: "Completed" if x == 0 else "Pending")
    merged["Folder"] = merged["folder_url"].fillna("#").apply(
        lambda u: f"<a href='{u}' target='_blank' rel='noopener noreferrer'>Open Folder</a>"
    )
    show_cols = ["Category", "App_Name", "reviewer", "Status", "missing", "total_rows", "approved", "denied", "changed", "Folder"]
    renamed = merged[show_cols].rename(columns={
        "App_Name": "App Name",
        "reviewer": "Reviewer",
        "missing": "Missing",
        "total_rows": "Total Rows",
        "approved": "Approved",
        "denied": "Denied",
        "changed": "Changed",
    })
    return renamed.to_html(index=False, escape=False)

def build_dashboard():

    container = widgets.VBox()

    btn_export = widgets.Button(description="💾 Save Reports", button_style='success', icon='file-excel')

    btn_global = widgets.Button(description="📊 Global Report", button_style='primary')

    lbl_out = widgets.Label()



    prog = _compute_overall_progress(stats, df)
    total_app_reviewers = prog["total_app_reviewers"]
    pending_app_reviewers = prog["pending_app_reviewers"]
    done_app_reviewers = prog["done_app_reviewers"]
    progress_pct = prog["progress_pct"]
    distinct_reviewers_to_notify = prog["distinct_reviewers_to_notify"]
    distinct_apps_remaining = prog["distinct_apps_remaining"]
    remaining_df = prog["remaining_work_df"]

    if not USE_WIDGETS:
        overview_html = _build_progress_overview_html(
            total_app_reviewers,
            pending_app_reviewers,
            done_app_reviewers,
            progress_pct,
            distinct_reviewers_to_notify,
            distinct_apps_remaining,
            remaining_df,
        )
        summary_html = _build_summary_table_html(stats, df)
        display(HTML(overview_html + "<h4>All App-Reviewers</h4>" + summary_html))
        return

    w_metrics = widgets.HTML(
        f"<b>Total app-reviewers:</b> {total_app_reviewers} &nbsp; "
        f"<b>Pending:</b> {pending_app_reviewers} &nbsp; "
        f"<b>Done:</b> {done_app_reviewers} &nbsp; "
        f"<b>Progress:</b> {progress_pct}% &nbsp; "
        f"<b>Distinct apps remaining:</b> {distinct_apps_remaining} &nbsp; "
        f"<b>Distinct reviewers remaining:</b> {distinct_reviewers_to_notify}"
    )
    w_bar = widgets.IntProgress(
        value=done_app_reviewers,
        min=0,
        max=max(1, total_app_reviewers),
        bar_style="success" if pending_app_reviewers == 0 else "info",
        layout=widgets.Layout(width="420px"),
    )
    w_bar_lbl = widgets.HTML(
        f"{done_app_reviewers}/{total_app_reviewers} completed",
        layout=widgets.Layout(width="180px"),
    )

    remaining_tables = _build_remaining_work_summary_tables(remaining_df)
    w_remaining_html = widgets.HTML(remaining_tables["app_html"])
    w_sort_label = widgets.HTML("<b>Sort by</b>", layout=widgets.Layout(margin="0 8px 0 0"))
    w_sort = widgets.ToggleButtons(
        options=[
            (f"App ({remaining_tables['app_count']})", "app"),
            (f"Reviewer ({remaining_tables['reviewer_count']})", "reviewer"),
        ],
        value="app",
        description="",
        layout=widgets.Layout(width="360px", margin="0"),
    )
    def _on_sort_change(change):
        if change.get("name") == "value":
            key = "app_html" if change.get("new") == "app" else "reviewer_html"
            w_remaining_html.value = remaining_tables[key]
    w_sort.observe(_on_sort_change, names="value")
    w_remaining_controls = widgets.HBox([w_sort_label, w_sort], layout=widgets.Layout(align_items="center", margin="0", padding="0"))
    w_remaining_box = widgets.VBox([w_remaining_controls, w_remaining_html], layout=widgets.Layout(margin="0", padding="0"))
    w_remaining_acc = widgets.Accordion(children=[w_remaining_box])
    w_remaining_acc.set_title(0, "🧾 Remaining Work (Pending App-Reviewers)")
    w_remaining_acc.selected_index = 0

    overview_box = widgets.VBox([
        widgets.HTML("<h4>📈 Progress Overview</h4>"),
        w_metrics,
        widgets.HBox([w_bar, w_bar_lbl], layout=widgets.Layout(align_items="center")),
        w_remaining_acc,
    ], layout=widgets.Layout(margin="8px 0 12px 0"))



    app_widgets = []

    for app_key in sorted(unified_data.keys()):

        app_data = unified_data[app_key]

        pct = int((app_data['stats']['completed_users'] / app_data['stats']['total_users'] * 100)) if app_data['stats']['total_users'] > 0 else 0

        w_lbl = widgets.HTML(f"<b>📂 {app_key}</b> &nbsp; <span style='background:#eee; padding:2px 5px; border-radius:4px'>{pct}% Done</span>", layout=widgets.Layout(width='400px'))

        w_stat = widgets.Dropdown(options=["Calculated", "Force Completed", "Action Required"], value=app_data['status_manual'], layout=widgets.Layout(width='150px'))

        widget_store[app_key] = {"data": app_data, "w_stat": w_stat}



        rev_list = widgets.VBox([

            widgets.HBox([

                widgets.HTML(f"<a href='{rd['folder_url']}' target='_blank'>{rn}</a>", layout=widgets.Layout(width='250px')),

                widgets.HTML(rd['status_calc'], layout=widgets.Layout(width='200px')),

                widgets.HTML(rd['detail_html'])

            ]) for rn, rd in app_data['reviewers'].items()

        ], layout=widgets.Layout(margin='5px 0 5px 20px', display='none'))



        btn_tog = widgets.Button(description="➕ Show Users", layout=widgets.Layout(width='100px'))

        def create_tog(w):

            def on_tog(b):

                if w.layout.display == 'none': w.layout.display = 'block'; b.description = "➖ Hide"

                else: w.layout.display = 'none'; b.description = "➕ Show Users"

            return on_tog

        btn_tog.on_click(create_tog(rev_list))

        app_widgets.append(widgets.VBox([widgets.HBox([btn_tog, w_lbl, w_stat]), rev_list]))



    def export(b):

        b.disabled=True; b.description="Saving..."

        saved_files = []

        for app_key, widget_data in widget_store.items():

            app_name = widget_data['data']['App_Name']

            category = widget_data['data']['Category']

            app_rows = df[(df['Category'] == category) & (df['App_Name'] == app_name)].to_dict('records')



            final_data = []

            for row in app_rows:

                overwrite_st = widget_data['w_stat'].value

                if overwrite_st == "Calculated": overwrite_st = ""



                final_data.append({

                    "User Name": row.get('user_name', ''),

                    "User Email": row.get('user_email', ''),

                    "Reviewer": row['reviewer'],

                    "File Name": row.get('file_name'),

                    "Reviewer Response": row.get('response'),

                    "Details of Access Change": row.get('details'),

                    "Overwrite Status": overwrite_st,

                    "Row Num": row.get('row_number'),

                    "Audit Log": row.get('Audit_Log')

                })



            if final_data:

                safe_name = re.sub(r'[\\/*?:"<>|]', "", app_name)

                f_path = safe_excel_path(f"{report_dir}/{safe_name}_{today_str}.xlsx")

                pd.DataFrame(final_data).to_excel(f_path, index=False)
                format_export_excel(f_path)

                saved_files.append(safe_name)



        lbl_out.value = f"Saved {len(saved_files)} files."

        b.disabled=False; b.description="💾 Save Reports"



    def export_global(b):

        b.disabled = True; b.description = "Saving..."

        prog = _compute_overall_progress(stats, df)

        rows = []

        for _, r in stats.sort_values("App_Name").iterrows():

            progress = "Completed" if r['missing'] == 0 else f"{int(r['missing'])}/{int(r['total_rows'])} missing"

            rows.append({

                "Category": r['Category'],

                "App Name": r['App_Name'],

                "Reviewer": r['reviewer'],

                "Progress": progress,

                "Total Approved": int(r['approved']),

                "Total Denied": int(r['denied']),

                "Total Changed": int(r['changed'])

            })

        f_path = safe_excel_path(f"{report_dir}/Summary_Report_{today_str}.xlsx")

        summary_df = pd.DataFrame(rows)
        overview_df = prog["overview_df"]
        remaining_df = prog["remaining_work_df"]

        with pd.ExcelWriter(f_path, engine="openpyxl") as writer:
            summary_df.to_excel(writer, index=False, sheet_name="Sheet1")
            overview_df.to_excel(writer, index=False, sheet_name="Progress", startrow=0)
            start = len(overview_df) + 2
            remaining_df.to_excel(writer, index=False, sheet_name="Progress", startrow=start)

        format_global_report_excel(f_path, remaining_header_row=start + 1)

        lbl_out.value = f"Global report saved."

        b.disabled = False; b.description = "📊 Global Report"



    btn_export.on_click(export)

    btn_global.on_click(export_global)

    container.children = tuple([widgets.HBox([btn_export, btn_global, lbl_out]), overview_box] + app_widgets)

    display(container)



if unified_data:

    build_dashboard()

else:

    print("⚠️ No data found.")


In [ ]:
# === CELL 7: Stage 7 — Email Notification Center ===
# Send review reminders to reviewers. Uses AD cache for email lookup (not per-reviewer Graph API).
# Restores the long-form reviewer preview UI while keeping newer runtime and lookup fixes.

import os
import json
import html
import ipywidgets as widgets
from urllib.parse import quote
from datetime import datetime, timedelta
import requests
import pandas as pd
import re
from IPython.display import display, clear_output, HTML

import importlib.util
from IPython import get_ipython


def _widgets_env_ready():
    try:
        ip = get_ipython()
        if ip is None or ip.__class__.__name__ != "ZMQInteractiveShell":
            return False
        has_nb = importlib.util.find_spec("widgetsnbextension") is not None
        has_lab = importlib.util.find_spec("jupyterlab_widgets") is not None
        return has_nb or has_lab
    except Exception:
        return False


USE_WIDGETS = _widgets_env_ready()


def parse_email_list(raw_text):
    if not raw_text:
        return []
    parts = re.split(r"[,\n;]+", str(raw_text))
    return [p.strip() for p in parts if p and p.strip()]


if "CHECKPOINT_DIR" not in globals():
    CHECKPOINT_DIR = os.path.join("output", datetime.now().strftime("%Y-%m-%d"), "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
if "INPUT_AD_CACHE_DIR" not in globals():
    INPUT_AD_CACHE_DIR = os.path.join("input", "ad_cache")
if "OUTPUT_AD_CACHE_DIR" not in globals():
    OUTPUT_AD_CACHE_DIR = os.path.join("output", datetime.now().strftime("%Y-%m-%d"), "ad_cache")
if "SENDER_EMAIL" not in globals():
    SENDER_EMAIL = os.getenv("SENDER_EMAIL", "").strip()

if "BRACKET_CHARS" not in globals():
    BRACKET_CHARS = set("[](){}<>")
if "KNOWN_DOMAIN_CORRECTIONS" not in globals():
    KNOWN_DOMAIN_CORRECTIONS = {"apple-bank.com": "applebank.com"}
if "EMAIL_PATTERN" not in globals():
    EMAIL_PATTERN = re.compile(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}$")

if "AER_EMAIL_TEMPLATE_FOOTER" not in globals():
    AER_EMAIL_TEMPLATE_FOOTER = "Sincerely,\nInformation Security Team"
if "AER_REVIEW_YEAR" not in globals():
    app_name = str(globals().get("APP_NAME", "")).strip()
    year_match = re.search(r"(20\d{2})", app_name)
    AER_REVIEW_YEAR = year_match.group(1) if year_match else str(datetime.now().year)

if "load_json_safe" not in globals():
    def load_json_safe(file_path):
        if not os.path.exists(file_path):
            return {}
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}


if "atomic_json_save" not in globals():
    def atomic_json_save(file_path, data):
        tmp_path = file_path + ".tmp"
        with open(tmp_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        os.replace(tmp_path, file_path)


if "is_email_valid" not in globals():
    def is_email_valid(email):
        return bool(EMAIL_PATTERN.match(str(email).strip().lower()))


if "correct_email_domain" not in globals():
    def correct_email_domain(email):
        email = str(email).strip().lower()
        for wrong, correct in KNOWN_DOMAIN_CORRECTIONS.items():
            if email.endswith(f"@{wrong}"):
                return email.replace(f"@{wrong}", f"@{correct}"), True
        return email, False


if "load_ad_cache" not in globals():
    def _find_latest_ad_cache():
        candidates = []
        for directory in [INPUT_AD_CACHE_DIR, OUTPUT_AD_CACHE_DIR]:
            if not os.path.exists(directory):
                continue
            for name in os.listdir(directory):
                if name.startswith("ad_users_") and name.endswith(".csv"):
                    candidates.append(os.path.join(directory, name))
        if not candidates:
            return None
        return max(candidates, key=os.path.getmtime)


    def load_ad_cache():
        path = _find_latest_ad_cache()
        if not path:
            return None, None, "No AD cache found."
        try:
            ad_df = pd.read_csv(path)
            required = {"email", "displayName"}
            missing = [column for column in required if column not in ad_df.columns]
            if missing:
                return None, path, f"AD cache missing columns: {missing}"
            return ad_df, path, ""
        except Exception as exc:
            return None, path, f"Failed to load AD cache: {exc}"


if "graph_post" not in globals():
    def graph_post(url, headers, json_payload=None, timeout=30):
        req_headers = dict(headers or {})
        if json_payload is not None:
            req_headers["Content-Type"] = "application/json"
        return requests.post(url, headers=req_headers, json=json_payload, timeout=timeout)


if "token_mgr" not in globals():
    class _NotebookTokenManager:
        def get_headers(self, scope_key="graph"):
            existing = globals().get("headers")
            if not existing:
                raise RuntimeError("Graph headers are not available. Run Cell 1 first.")
            return existing


    token_mgr = _NotebookTokenManager()


EMAIL_CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, "email_sent.json")
STAGE7_DEFAULTS_FILENAME = "email_defaults.json"
STAGE7_DEFAULTS_ENV_MAP = {
    "subject": "AER_EMAIL_SUBJECT",
    "reply_to": "AER_EMAIL_REPLY_TO",
    "cc": "AER_EMAIL_CC",
    "body_before_chart": "AER_EMAIL_BODY_BEFORE_CHART",
    "body_after_chart": "AER_EMAIL_BODY_AFTER_CHART",
}


def _stage7_normalize_text(value):
    if value is None:
        return ""
    if isinstance(value, list):
        value = "\n".join(str(item) for item in value)
    return str(value).replace("\r\n", "\n").replace("\r", "\n").replace("\\n", "\n").strip()


def _stage7_normalize_email_field(value):
    if value is None:
        return ""
    if isinstance(value, list):
        raw_text = ", ".join(str(item) for item in value)
    else:
        raw_text = str(value).replace("\\n", "\n")
    return ", ".join(parse_email_list(raw_text))


def _stage7_text_to_html(text):
    normalized = _stage7_normalize_text(text)
    if not normalized:
        return ""

    mailto_placeholders = {}

    def _preserve_mailto_anchor(match):
        placeholder = f"__STAGE7_MAILTO_LINK_{len(mailto_placeholders)}__"
        href = html.escape(match.group("href").strip(), quote=True)
        label = html.escape(match.group("label").strip())
        if not label:
            label = href
        mailto_placeholders[placeholder] = f"<a href=\"{href}\">{label}</a>"
        return placeholder

    anchor_pattern = re.compile(
        r"<a\s+(?:href|herf)\s*=\s*['\"](?P<href>mailto:[^'\"]+)['\"]\s*>(?P<label>.*?)</a>",
        re.IGNORECASE | re.DOTALL,
    )
    rendered = anchor_pattern.sub(_preserve_mailto_anchor, normalized)
    rendered = html.escape(rendered).replace("\n", "<br>")
    for placeholder, anchor_html in mailto_placeholders.items():
        rendered = rendered.replace(placeholder, anchor_html)
    return rendered


def _stage7_candidate_defaults_paths():
    candidates = []
    env_path = os.getenv("AER_EMAIL_DEFAULTS_FILE", "").strip()
    if env_path:
        candidates.append(env_path)
    cwd = os.getcwd()
    for path in [
        os.path.join(cwd, STAGE7_DEFAULTS_FILENAME),
        os.path.join(cwd, "REVIEW-REPORT", STAGE7_DEFAULTS_FILENAME),
    ]:
        if path not in candidates:
            candidates.append(path)
    return candidates


def _stage7_load_json_defaults():
    for path in _stage7_candidate_defaults_paths():
        if not path or not os.path.exists(path):
            continue
        try:
            with open(path, "r", encoding="utf-8") as f:
                payload = json.load(f)
        except Exception as exc:
            return {}, path, f"Failed to load email defaults JSON: {exc}"
        if not isinstance(payload, dict):
            return {}, path, "Email defaults JSON must contain a JSON object."
        return payload, path, ""
    return {}, "", ""


def _stage7_build_defaults():
    defaults = {
        "subject": f"REMINDER - {AER_REVIEW_YEAR} Entitlement Review Update",
        "reply_to": "",
        "cc": "",
        "body_before_chart": "",
        "body_after_chart": (
            "Please complete these reviews as soon as possible. "
            "Your prompt assistance to this matter is appreciated."
        ),
    }

    for key, env_name in STAGE7_DEFAULTS_ENV_MAP.items():
        if env_name not in os.environ:
            continue
        raw_value = os.getenv(env_name, "")
        if key in {"reply_to", "cc"}:
            defaults[key] = _stage7_normalize_email_field(raw_value)
        else:
            defaults[key] = _stage7_normalize_text(raw_value)

    json_defaults, json_path, json_warning = _stage7_load_json_defaults()
    for key in STAGE7_DEFAULTS_ENV_MAP:
        if key not in json_defaults:
            continue
        raw_value = json_defaults.get(key)
        if key in {"reply_to", "cc"}:
            defaults[key] = _stage7_normalize_email_field(raw_value)
        else:
            defaults[key] = _stage7_normalize_text(raw_value)

    return defaults, json_path, json_warning


STAGE7_EMAIL_DEFAULTS, STAGE7_DEFAULTS_PATH, STAGE7_DEFAULTS_WARNING = _stage7_build_defaults()


def _stage7_defaults_output_path():
    if STAGE7_DEFAULTS_PATH:
        return STAGE7_DEFAULTS_PATH
    cwd = os.getcwd()
    if os.path.basename(cwd) == "REVIEW-REPORT":
        return os.path.join(cwd, STAGE7_DEFAULTS_FILENAME)
    review_report_dir = os.path.join(cwd, "REVIEW-REPORT")
    if os.path.isdir(review_report_dir):
        return os.path.join(review_report_dir, STAGE7_DEFAULTS_FILENAME)
    return os.path.join(cwd, STAGE7_DEFAULTS_FILENAME)


STAGE7_DEFAULTS_OUTPUT_PATH = _stage7_defaults_output_path()


def _stage7_current_default_value(widget_name, config_key):
    widget = globals().get(widget_name)
    if widget is not None and hasattr(widget, "value"):
        raw_value = widget.value
    else:
        raw_value = STAGE7_EMAIL_DEFAULTS.get(config_key, "")
    if config_key in {"reply_to", "cc"}:
        return _stage7_normalize_email_field(raw_value)
    return _stage7_normalize_text(raw_value)


def _stage7_collect_current_defaults():
    return {
        "subject": _stage7_current_default_value("w_subject_tpl", "subject"),
        "reply_to": _stage7_current_default_value("w_reply_to", "reply_to"),
        "cc": _stage7_current_default_value("w_cc_global", "cc"),
        "body_before_chart": _stage7_current_default_value("w_body_before_chart", "body_before_chart"),
        "body_after_chart": _stage7_current_default_value("w_body_after_chart", "body_after_chart"),
    }


def _stage7_save_current_defaults(path=None):
    target_path = path or STAGE7_DEFAULTS_OUTPUT_PATH
    os.makedirs(os.path.dirname(target_path), exist_ok=True)
    atomic_json_save(target_path, _stage7_collect_current_defaults())
    return target_path


def _normalize_reviewer_name(value):
    if pd.isna(value):
        return ""
    cleaned = re.sub(r"[^\w\s]", "", str(value).strip().lower())
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned


def _build_identity_index_cell7(df):
    email_set = set()
    name_to_emails = {}
    if df is None:
        return email_set, name_to_emails
    for _, row in df.iterrows():
        email = str(row.get("email", "")).strip().lower()
        if not email or email == "nan":
            continue
        email_set.add(email)
        name = _normalize_reviewer_name(row.get("displayName", ""))
        if name:
            name_to_emails.setdefault(name, set()).add(email)
    return email_set, name_to_emails


def _resolve_identity_cell7(value, ad_email_set, ad_name_map):
    """Resolve reviewer to email with punctuation-insensitive name matching."""
    if pd.isna(value):
        return False, "", "Value is blank/NaN"
    raw = str(value).strip()
    if not raw:
        return False, "", "Value is blank"
    if any(ch in raw for ch in BRACKET_CHARS):
        return False, "", "Value contains bracket characters"
    lower_val = raw.lower()
    if "@" in lower_val:
        if not is_email_valid(lower_val):
            return False, "", "Email format is invalid"
        if lower_val not in ad_email_set:
            corrected, was_corrected = correct_email_domain(lower_val)
            if was_corrected and corrected in ad_email_set:
                return True, corrected, ""
            return False, "", "Email not found in AD"
        return True, lower_val, ""
    normalized = _normalize_reviewer_name(raw)
    matches = sorted(list(ad_name_map.get(normalized, set())))
    if len(matches) == 0:
        return False, "", "Name not found in AD"
    if len(matches) > 1:
        return False, "", f"Name maps to {len(matches)} AD accounts"
    return True, matches[0], ""


def _email_lookup_from_ad(reviewer_name, ad_email_set=None, ad_name_map=None):
    """Resolve reviewer name to email using AD cache (no Graph API call)."""
    if ad_email_set is None or ad_name_map is None:
        ad_df, _, _ = load_ad_cache()
        if ad_df is not None:
            ad_email_set, ad_name_map = _build_identity_index_cell7(ad_df)
        else:
            return ""
    ok, email, _ = _resolve_identity_cell7(reviewer_name, ad_email_set, ad_name_map)
    return email if ok else ""


def get_email(name):
    return _email_lookup_from_ad(name)


def fmt_date_long(iso_date_str):
    if not iso_date_str or pd.isna(iso_date_str):
        return "Unknown Date"
    try:
        dt_str = str(iso_date_str).replace("Z", "").split(".")[0]
        dt = datetime.fromisoformat(dt_str)
        return dt.strftime("%B %d, %Y")
    except Exception:
        return str(iso_date_str)


def calc_due_date_long(iso_date_str):
    if not iso_date_str or pd.isna(iso_date_str):
        return "ASAP"
    try:
        dt_str = str(iso_date_str).replace("Z", "").split(".")[0]
        dt = datetime.fromisoformat(dt_str)
        due = dt + timedelta(days=14)
        return due.strftime("%B %d, %Y")
    except Exception:
        return "ASAP"


def iso_to_date(iso_date_str):
    if not iso_date_str or pd.isna(iso_date_str):
        return None
    try:
        dt_str = str(iso_date_str).replace("Z", "").split(".")[0]
        dt = datetime.fromisoformat(dt_str)
        return dt.date()
    except Exception:
        return None


def format_date_long(date_obj):
    if not date_obj:
        return ""
    try:
        return date_obj.strftime("%B %d, %Y")
    except Exception:
        return str(date_obj)


def infer_section(file_date_raw):
    if not file_date_raw or pd.isna(file_date_raw):
        return "followup"
    try:
        dt_str = str(file_date_raw).replace("Z", "").split(".")[0]
        sent_dt = datetime.fromisoformat(dt_str)
        due_dt = sent_dt + timedelta(days=14)
        return "new" if due_dt >= datetime.now() else "followup"
    except Exception:
        return "followup"


def build_section_table(rows):
    ordered_rows = sorted(rows, key=lambda x: (str(x.get("Category", "")), str(x.get("App_Name", ""))))
    cell_style = "padding:6px; overflow-wrap:anywhere; word-break:break-word; white-space:normal; vertical-align:middle"
    table = (
        "<div style='width:100%; text-align:center'>"
        "<table border='1' style='margin:0 auto; border-collapse:collapse; width:96%; font-size:12px; "
        "border:1px solid #d8dde6; table-layout:fixed'>"
    )
    table += (
        "<tr style='background:#eef3f8; font-weight:bold'>"
        "<th style='padding:6px; width:18%; vertical-align:middle'>Quarter</th>"
        "<th style='padding:6px; width:22%; vertical-align:middle'>App Name</th>"
        "<th style='padding:6px; width:18%; vertical-align:middle'>Sent Date</th>"
        "<th style='padding:6px; width:18%; vertical-align:middle'>Due Date</th>"
        "<th style='padding:6px; width:14%; vertical-align:middle'>Users to review</th>"
        "<th style='padding:6px; width:10%; vertical-align:middle'>Link</th>"
        "</tr>"
    )
    for row in ordered_rows:
        link_url = html.escape(str(row.get("folder_url") or "#"), quote=True)
        link_html = (
            f"<a href='{link_url}' target='_blank' rel='noopener noreferrer' "
            "style='background:#0078d4;color:#fff;padding:6px 10px;border-radius:6px;"
            "text-decoration:none;display:inline-block;white-space:nowrap'>Open Folder</a>"
        )
        table += (
            "<tr>"
            f"<td style='{cell_style}'>{html.escape(str(row.get('Category', '')))}</td>"
            f"<td style='{cell_style}'>{html.escape(str(row.get('App_Name', '')))}</td>"
            f"<td style='padding:6px; vertical-align:middle'>{html.escape(str(row.get('sent_date', '')))}</td>"
            f"<td style='padding:6px; vertical-align:middle'>{html.escape(str(row.get('due_date', '')))}</td>"
            f"<td style='padding:6px; text-align:center; vertical-align:middle'>{html.escape(str(row.get('missing', '')))}</td>"
            f"<td style='padding:6px; text-align:center; vertical-align:middle'>{link_html}</td>"
            "</tr>"
        )
    table += "</table></div>"
    return table


def _build_followup_progress_email_html():
    stats_df = globals().get("stats")
    raw_df = globals().get("df")
    progress_func = globals().get("_compute_overall_progress")
    if stats_df is None or raw_df is None or not callable(progress_func):
        return ""
    try:
        prog = progress_func(stats_df, raw_df)
    except Exception:
        return ""
    total_app_reviewers = prog["total_app_reviewers"]
    pending_app_reviewers = prog["pending_app_reviewers"]
    done_app_reviewers = prog["done_app_reviewers"]
    progress_pct = prog["progress_pct"]
    distinct_reviewers_to_notify = prog["distinct_reviewers_to_notify"]
    bar_pct = 0.0 if total_app_reviewers == 0 else round(done_app_reviewers / total_app_reviewers * 100, 1)
    remaining_pct = max(0.0, 100.0 - bar_pct)
    pct_text = f"{progress_pct:g}"
    top_line = (
        f"<b>Total app-reviewers:</b> {total_app_reviewers} &nbsp; "
        f"<b>Pending:</b> {pending_app_reviewers} &nbsp; "
        f"<b>Done:</b> {done_app_reviewers} &nbsp; "
        f"<b>Progress:</b> {pct_text}%"
    )
    reviewer_line = f"<b>Distinct reviewers remaining: {distinct_reviewers_to_notify}</b>"
    bar_line = (
        "<div style='width:420px; height:12px; border:1px solid #cfd7e6; border-radius:6px; overflow:hidden; display:inline-flex; vertical-align:middle'>"
        f"<div style='height:12px; width:{bar_pct}%; background:#4caf50'></div>"
        f"<div style='height:12px; width:{remaining_pct}%; background:#f04438'></div>"
        "</div>"
        f"<span style='margin-left:8px; font-size:12px; color:#555'>{done_app_reviewers}/{total_app_reviewers} completed ({pct_text}%)</span>"
    )
    return f"{top_line}<br>{reviewer_line}<br>{bar_line}<br><br>"


def build_email_html(reviewer_name, new_rows, followup_rows):
    before_chart_html = _stage7_text_to_html(_stage7_current_default_value("w_body_before_chart", "body_before_chart"))
    after_chart_html = _stage7_text_to_html(_stage7_current_default_value("w_body_after_chart", "body_after_chart"))
    footer_html = _stage7_text_to_html(AER_EMAIL_TEMPLATE_FOOTER)

    parts = [
        f"Hi {html.escape(str(reviewer_name))},<br><br>",
        "The Information Security team is sending you an entitlement review update.<br><br>",
    ]

    if before_chart_html:
        parts.append(before_chart_html)
        parts.append("<br><br>")

    if new_rows:
        parts.append("<b>- New Applications to Review</b><br>")
        parts.append(build_section_table(new_rows))
        parts.append("<br><br>")

    if followup_rows:
        parts.append("<b>- Follow-up on Remaining Reviews</b><br>")
        parts.append("<br>")
        parts.append(_build_followup_progress_email_html())
        parts.append(build_section_table(followup_rows))
        parts.append("<br><br>")

    if not new_rows and not followup_rows:
        parts.append("No applications are selected for this update.<br><br>")

    if after_chart_html:
        parts.append(after_chart_html)
        parts.append("<br><br>")

    if footer_html:
        parts.append(footer_html)

    return "".join(parts)


def resolve_subject(template, reviewer_name):
    try:
        return template.format(reviewer_name=reviewer_name)
    except Exception:
        return template


def _build_email_fallback_html(df_raw):
    if df_raw is None or df_raw.empty:
        return "<i>No pending emails to display.</i>"
    df_show = df_raw.copy()
    df_show["Folder"] = df_show["folder_url"].fillna("#").apply(
        lambda url: (
            f"<a href='{html.escape(str(url), quote=True)}' target='_blank' rel='noopener noreferrer' "
            "style='background:#0078d4;color:#fff;padding:6px 10px;border-radius:6px;"
            "text-decoration:none;display:inline-block;white-space:nowrap'>Open Folder</a>"
        )
    )
    show_cols = ["Category", "App_Name", "reviewer", "missing", "sent_date", "due_date", "email", "Folder"]
    renamed = df_show[show_cols].rename(columns={
        "App_Name": "App Name",
        "reviewer": "Reviewer",
        "missing": "Missing",
        "sent_date": "Sent Date",
        "due_date": "Due Date",
        "email": "Email",
    })
    return renamed.to_html(index=False, escape=False)


raw_df = globals().get("r6_df")
if raw_df is None:
    raw_df = globals().get("df")

if raw_df is None or raw_df.empty:
    display(widgets.HTML("<h4 style='color:red'>⚠️ No data found. Run the report scan first.</h4>"))
else:
    pending_df = raw_df[raw_df["is_missing"] == True].copy()
    targets = pending_df.groupby(["Category", "App_Name", "reviewer", "folder_url"]).agg(
        missing_count=("is_missing", "count"),
        file_date_raw=("File_Created_Date", "min"),
    ).reset_index()

    notes_db = manual_data_store if "manual_data_store" in globals() else {}
    print("🔍 Looking up emails for pending reviewers...")
    email_cache = {}
    raw_data_list = []

    ad_df, _, _ = load_ad_cache()
    ad_email_set, ad_name_map = _build_identity_index_cell7(ad_df) if ad_df is not None else (set(), {})

    for _, row in targets.iterrows():
        app_key = f"{row['Category']} > {row['App_Name']}"
        reviewer_key = row["reviewer"]
        app_node = notes_db.get(app_key, {})
        if app_node.get("app_status") in ["Force Completed", "N/A"]:
            continue

        rev_override = app_node.get("reviewers", {}).get(reviewer_key, {}).get("override", "-")
        if rev_override in ["Mark Done", "Waived", "Escalated"]:
            continue

        if reviewer_key not in email_cache:
            email_cache[reviewer_key] = _email_lookup_from_ad(reviewer_key, ad_email_set, ad_name_map)

        raw_data_list.append(
            {
                "Category": row["Category"],
                "App_Name": row["App_Name"],
                "app_key": app_key,
                "reviewer": reviewer_key,
                "missing": int(row["missing_count"]),
                "folder_url": row["folder_url"],
                "sent_date": fmt_date_long(row["file_date_raw"]),
                "due_date": calc_due_date_long(row["file_date_raw"]),
                "file_date_raw": row["file_date_raw"],
                "email": email_cache[reviewer_key],
            }
        )

    clear_output()

    default_subject = STAGE7_EMAIL_DEFAULTS.get("subject") or f"REMINDER - {AER_REVIEW_YEAR} Entitlement Review Update"
    defaults_status_text = ""
    defaults_status_color = "#666"
    if STAGE7_DEFAULTS_WARNING:
        defaults_status_text = STAGE7_DEFAULTS_WARNING
        defaults_status_color = "#b42318"
    elif STAGE7_DEFAULTS_PATH:
        defaults_status_text = f"Loaded email defaults from {os.path.basename(STAGE7_DEFAULTS_PATH)}"

    df_raw = pd.DataFrame(raw_data_list)
    row_logic_store = {}
    app_section_store = {}
    app_send_store = {}
    app_due_store = {}
    app_due_mode = {}
    app_send_mode = {}
    app_send_guard = {}
    app_guard = {}
    app_selector_rows = []
    global_observers_bound = {"bound": False}

    w_subject_tpl = widgets.Text(
        value=default_subject,
        placeholder="Supports {reviewer_name}",
        layout=widgets.Layout(width="100%"),
    )
    w_cc_global = widgets.Textarea(
        value=STAGE7_EMAIL_DEFAULTS.get("cc", ""),
        placeholder="manager@applebank.com, team@applebank.com",
        layout=widgets.Layout(width="100%", height="70px"),
    )
    w_reply_to = widgets.Text(
        value=STAGE7_EMAIL_DEFAULTS.get("reply_to", ""),
        placeholder="security-team@applebank.com",
        layout=widgets.Layout(width="100%"),
    )
    w_body_before_chart = widgets.Textarea(
        value=STAGE7_EMAIL_DEFAULTS.get("body_before_chart", ""),
        placeholder="Optional text shown before the app tables",
        layout=widgets.Layout(width="100%", height="90px"),
    )
    w_body_after_chart = widgets.Textarea(
        value=STAGE7_EMAIL_DEFAULTS.get("body_after_chart", ""),
        placeholder="Optional text shown after the app tables",
        layout=widgets.Layout(width="100%", height="110px"),
    )
    w_send_from = widgets.Text(
        value=(SENDER_EMAIL or ""),
        placeholder="shared-mailbox@applebank.com",
        layout=widgets.Layout(width="100%"),
    )
    w_global_send = widgets.DatePicker(
        value=None,
        description="Global Send",
        layout=widgets.Layout(width="240px"),
    )
    w_global_send.style.description_width = "95px"
    w_global_due = widgets.DatePicker(
        value=None,
        description="Global Due",
        layout=widgets.Layout(width="240px"),
    )
    w_global_due.style.description_width = "95px"
    btn_refresh = widgets.Button(
        description="🔄 Refresh List",
        button_style="info",
        layout=widgets.Layout(width="180px"),
    )
    btn_send_all = widgets.Button(
        description="🔥 Send All (Batch)",
        button_style="danger",
        layout=widgets.Layout(width="200px"),
    )
    btn_save_defaults = widgets.Button(
        description="💾 Save Defaults JSON",
        button_style="success",
        layout=widgets.Layout(width="180px"),
    )
    defaults_status = widgets.HTML(
        value=(
            f"<div style='font-size:11px;color:{defaults_status_color};margin-top:4px'>"
            f"{html.escape(defaults_status_text)}</div>"
        ) if defaults_status_text else ""
    )
    defaults_save_status = widgets.HTML()

    app_selector_box = widgets.VBox()
    reviewer_box = widgets.VBox()

    def derive_dates_for_row(row):
        app_key = row.get("app_key")
        send_widget = app_send_store.get(app_key)
        due_widget = app_due_store.get(app_key)

        send_override = send_widget.value if send_widget else None
        due_override = due_widget.value if due_widget else None

        sent_date = send_override or iso_to_date(row.get("file_date_raw"))
        sent_str = format_date_long(sent_date) if sent_date else "Unknown Date"

        due_date = due_override
        if not due_date and sent_date:
            due_date = sent_date + timedelta(days=14)
        due_str = format_date_long(due_date) if due_date else "ASAP"
        return sent_str, due_str


    def with_derived_dates(rows):
        out = []
        for row in rows:
            sent_str, due_str = derive_dates_for_row(row)
            row_copy = dict(row)
            row_copy["sent_date"] = sent_str
            row_copy["due_date"] = due_str
            out.append(row_copy)
        return out


    def build_app_selector_state():
        nonlocal_df = df_raw.copy()
        if nonlocal_df.empty:
            app_selector_box.children = (widgets.HTML("<i>No apps available.</i>"),)
            return

        app_summary = nonlocal_df.groupby(["app_key", "Category", "App_Name"]).agg(
            reviewer_count=("reviewer", "nunique"),
            missing_total=("missing", "sum"),
            file_date_raw=("file_date_raw", "min"),
        ).reset_index().sort_values(["Category", "App_Name"])

        current_section_values = {key: widget.value for key, widget in app_section_store.items()}
        current_send_values = {key: widget.value for key, widget in app_send_store.items()}
        current_due_values = {key: widget.value for key, widget in app_due_store.items()}
        current_due_mode = dict(app_due_mode)
        current_send_mode = dict(app_send_mode)

        app_section_store.clear()
        app_send_store.clear()
        app_due_store.clear()
        app_due_mode.clear()
        app_send_mode.clear()
        app_send_guard.clear()
        app_guard.clear()

        max_label_chars = 0
        for _, row in app_summary.iterrows():
            label = f"Quarter: {row['Category']} > {row['App_Name']}"
            max_label_chars = max(max_label_chars, len(label))
        info_basis_px = max(320, min(560, max_label_chars * 7))

        rows = []

        for _, row in app_summary.iterrows():
            app_key = row["app_key"]
            default_value = current_section_values.get(app_key, infer_section(row["file_date_raw"]))
            w_section = widgets.Dropdown(
                options=[("New", "new"), ("Follow-up", "followup"), ("Skip", "skip")],
                value=default_value,
                layout=widgets.Layout(width="140px"),
            )
            app_section_store[app_key] = w_section

            if app_key in current_send_values:
                send_default = current_send_values[app_key]
            else:
                send_default = w_global_send.value if w_global_send.value is not None else iso_to_date(row["file_date_raw"])

            if app_key in current_due_values:
                due_default = current_due_values[app_key]
            else:
                if w_global_due.value is not None:
                    due_default = w_global_due.value
                elif send_default:
                    due_default = send_default + timedelta(days=14)
                else:
                    due_default = None

            w_send = widgets.DatePicker(value=send_default, layout=widgets.Layout(width="160px"))
            w_send.description = "Send"
            w_send.style.description_width = "45px"

            w_due = widgets.DatePicker(value=due_default, layout=widgets.Layout(width="160px"))
            w_due.description = "Due"
            w_due.style.description_width = "45px"

            app_send_store[app_key] = w_send
            app_due_store[app_key] = w_due
            app_due_mode[app_key] = current_due_mode.get(app_key, "auto")
            app_send_mode[app_key] = current_send_mode.get(app_key, "auto")
            app_guard[app_key] = False
            app_send_guard[app_key] = False

            info_html = (
                "<div style='overflow-wrap:anywhere; word-break:break-word; white-space:normal'>"
                f"<b>Quarter: {html.escape(str(row['Category']))} &gt; {html.escape(str(row['App_Name']))}</b>"
                f"<span style='color:#555; margin-left:12px; display:inline-block'>Reviewers: {int(row['reviewer_count'])}</span>"
                f"<span style='color:#555; margin-left:12px; display:inline-block'>Pending Items: {int(row['missing_total'])}</span>"
                "</div>"
            )
            w_info = widgets.HTML(
                info_html,
                layout=widgets.Layout(flex=f"1 1 {info_basis_px}px", min_width="260px"),
            )
            w_controls = widgets.HBox(
                [w_section, w_send, w_due],
                layout=widgets.Layout(flex_flow="row wrap", align_items="center", justify_content="flex-end"),
            )
            rows.append(
                widgets.HBox(
                    [w_info, w_controls],
                    layout=widgets.Layout(
                        border="1px solid #ececec",
                        padding="6px",
                        margin="2px 0",
                        width="100%",
                        flex_flow="row wrap",
                        align_items="center",
                        justify_content="space-between",
                    ),
                )
            )

        app_selector_rows[:] = rows
        app_selector_box.children = (
            widgets.HTML(
                "<div style='font-size:12px;color:#555'>Set each app once here. The selection applies to <b>all reviewers</b>.</div>"
            ),
            widgets.VBox(rows, layout=widgets.Layout(width="100%")),
        )


    def get_sectioned_rows(reviewer_rows):
        new_rows, followup_rows = [], []
        for row in reviewer_rows:
            selected_section = app_section_store.get(row["app_key"])
            section_value = selected_section.value if selected_section else "followup"
            if section_value == "new":
                new_rows.append(row)
            elif section_value == "followup":
                followup_rows.append(row)
        return with_derived_dates(new_rows), with_derived_dates(followup_rows)


    def update_preview(key):
        state = row_logic_store[key]
        new_rows, followup_rows = get_sectioned_rows(state["rows"])
        new_missing = sum(row["missing"] for row in new_rows)
        followup_missing = sum(row["missing"] for row in followup_rows)
        state["w_summary"].value = (
            f"<div style='color:#0b6'><b>New:</b> {len(new_rows)} app(s), {new_missing} item(s)</div>"
            f"<div style='color:#9a6700'><b>Follow-up:</b> {len(followup_rows)} app(s), {followup_missing} item(s)</div>"
        )
        body_html = build_email_html(state["reviewer"], new_rows, followup_rows)
        state["body_html"] = body_html
        state["w_preview"].value = (
            "<div style='font-family:sans-serif; font-size:12px; line-height:1.45; padding:8px; color:#333'>"
            f"{body_html}"
            "</div>"
        )


    def refresh_all_previews(change=None):
        for key in list(row_logic_store.keys()):
            update_preview(key)


    def build_reviewer_cards():
        row_logic_store.clear()

        if df_raw.empty:
            reviewer_box.children = (widgets.HTML("<h3>✅ No pending emails to send!</h3>"),)
            return

        sent_log = load_json_safe(EMAIL_CHECKPOINT_FILE)
        grouped = df_raw.sort_values(["reviewer", "Category", "App_Name"]).groupby(["reviewer", "email"], dropna=False)
        cards = []

        for idx, ((reviewer, email), group) in enumerate(grouped):
            key = f"{reviewer}_{idx}"
            reviewer_rows = group.to_dict("records")
            already_sent = sent_log.get(key, {}).get("sent", False)
            sent_stamp = sent_log.get(key, {}).get("ts", "")
            email_color = "#0078d4" if email else "red"

            w_chk = widgets.Checkbox(value=not already_sent, layout=widgets.Layout(width="30px"))
            w_email = widgets.Text(value=email or "", placeholder="Email", layout=widgets.Layout(width="100%"))
            w_btn = widgets.Button(
                description="Sent" if already_sent else "🚀 Send",
                button_style="success" if already_sent else "warning",
                layout=widgets.Layout(width="90px"),
                disabled=already_sent,
            )
            w_summary = widgets.HTML(layout=widgets.Layout(width="100%"))
            w_preview = widgets.HTML(
                value="",
                layout=widgets.Layout(width="100%", border="1px solid #d8dde6"),
            )
            w_detail = widgets.HTML(
                value=(
                    f"<b>👤 {html.escape(str(reviewer))}</b><br>"
                    f"<span style='color:{email_color}'>{html.escape(str(email or '(No Email)'))}</span><br>"
                    f"<span style='color:#666'>Fetched Apps: {len(reviewer_rows)}</span>"
                    + (
                        f"<br><span style='color:#2e7d32'>✅ Sent {html.escape(sent_stamp[:19].replace('T', ' '))}</span>"
                        if already_sent and sent_stamp
                        else ("<br><span style='color:#2e7d32'>✅ Sent</span>" if already_sent else "")
                    )
                ),
                layout=widgets.Layout(min_width="240px", flex="1 1 240px"),
            )

            row_logic_store[key] = {
                "reviewer": reviewer,
                "rows": reviewer_rows,
                "w_chk": w_chk,
                "w_email": w_email,
                "w_btn": w_btn,
                "w_summary": w_summary,
                "w_preview": w_preview,
                "body_html": "",
            }

            def make_sender(k):
                def on_send(_):
                    state = row_logic_store[k]
                    target_email = state["w_email"].value.strip()
                    if not target_email:
                        state["w_btn"].button_style = "danger"
                        state["w_btn"].description = "No Email"
                        return

                    new_rows, followup_rows = get_sectioned_rows(state["rows"])
                    if not new_rows and not followup_rows:
                        state["w_btn"].button_style = "danger"
                        state["w_btn"].description = "No Apps"
                        return

                    state["w_btn"].disabled = True
                    state["w_btn"].description = "..."
                    try:
                        update_preview(k)
                        final_subject = resolve_subject(w_subject_tpl.value, state["reviewer"])
                        sender_mailbox = (w_send_from.value or "").strip() or (SENDER_EMAIL or "").strip()
                        if not sender_mailbox:
                            state["w_btn"].button_style = "danger"
                            state["w_btn"].description = "No Sender"
                            return

                        headers_graph = token_mgr.get_headers("graph")
                        url = f"https://graph.microsoft.com/v1.0/users/{quote(sender_mailbox)}/sendMail"
                        to_list = [{"emailAddress": {"address": target_email}}]
                        cc_list = [{"emailAddress": {"address": email}} for email in parse_email_list(w_cc_global.value)]
                        reply_to_list = [{"emailAddress": {"address": email}} for email in parse_email_list(w_reply_to.value)]

                        message_payload = {
                            "subject": final_subject,
                            "body": {"contentType": "HTML", "content": state["body_html"]},
                            "toRecipients": to_list,
                        }
                        if cc_list:
                            message_payload["ccRecipients"] = cc_list
                        if reply_to_list:
                            message_payload["replyTo"] = reply_to_list

                        response = graph_post(url, headers_graph, json_payload={"message": message_payload})
                        if response.status_code == 202:
                            state["w_btn"].button_style = "success"
                            state["w_btn"].description = "Sent"
                            state["w_btn"].disabled = True
                            state["w_chk"].value = False
                            checkpoint_state = load_json_safe(EMAIL_CHECKPOINT_FILE)
                            checkpoint_state[k] = {
                                "sent": True,
                                "ts": datetime.now().isoformat(),
                                "email": target_email,
                                "subject": final_subject,
                            }
                            atomic_json_save(EMAIL_CHECKPOINT_FILE, checkpoint_state)
                        else:
                            state["w_btn"].button_style = "danger"
                            state["w_btn"].description = "Fail"
                            print(response.text[:400])
                    except Exception as exc:
                        print(exc)
                        state["w_btn"].button_style = "danger"
                        state["w_btn"].description = "Err"
                    finally:
                        if state["w_btn"].description != "Sent":
                            state["w_btn"].disabled = False

                return on_send

            sender_func = make_sender(key)
            w_btn.on_click(sender_func)
            row_logic_store[key]["trigger_send"] = sender_func

            header_row = widgets.HBox(
                [
                    w_chk,
                    w_detail,
                    widgets.VBox(
                        [
                            widgets.HTML("<b>Summary</b>"),
                            w_summary,
                        ],
                        layout=widgets.Layout(flex="2 1 320px", min_width="260px", padding="0 12px"),
                    ),
                    widgets.VBox(
                        [
                            widgets.HTML("<b>Send To</b>"),
                            w_email,
                            w_btn,
                        ],
                        layout=widgets.Layout(flex="1 1 220px", min_width="220px"),
                    ),
                ],
                layout=widgets.Layout(align_items="flex-start", flex_flow="row wrap"),
            )

            cards.append(
                widgets.VBox(
                    [
                        header_row,
                        widgets.HTML("<div style='margin:8px 0 4px 0'><b>Email Preview</b></div>"),
                        w_preview,
                    ],
                    layout=widgets.Layout(border="1px solid #cfd8e3", margin="8px 0", padding="10px"),
                )
            )

            update_preview(key)

        reviewer_box.children = tuple(cards)


    def set_all_app_sections(value):
        for dropdown in app_section_store.values():
            dropdown.value = value
        refresh_all_previews()


    def trigger_batch_send(_):
        btn_send_all.disabled = True
        btn_send_all.description = "Sending..."
        count = 0
        for key, state in row_logic_store.items():
            if state["w_chk"].value and state["w_btn"].description != "Sent":
                state["trigger_send"](state["w_btn"])
                count += 1
        btn_send_all.disabled = False
        btn_send_all.description = f"🔥 Send All (Batch) - Done {count}"


    def save_defaults_json(_):
        btn_save_defaults.disabled = True
        btn_save_defaults.description = "Saving..."
        try:
            target_path = _stage7_save_current_defaults()
            defaults_save_status.value = (
                "<div style='font-size:11px;color:#067647'>"
                f"Saved defaults to {html.escape(os.path.basename(target_path))}"
                "</div>"
            )
        except Exception as exc:
            defaults_save_status.value = (
                "<div style='font-size:11px;color:#b42318'>"
                f"{html.escape(str(exc))}"
                "</div>"
            )
        finally:
            btn_save_defaults.disabled = False
            btn_save_defaults.description = "💾 Save Defaults JSON"


    def render_all(_=None):
        build_app_selector_state()

        for dropdown in app_section_store.values():
            dropdown.observe(refresh_all_previews, names="value")

        for app_key, send_picker in app_send_store.items():
            def _on_send(change, app_key=app_key):
                if change.get("name") != "value":
                    return
                if app_send_guard.get(app_key):
                    return

                app_send_mode[app_key] = "manual"

                if app_due_mode.get(app_key, "auto") == "auto":
                    send_val = change.get("new")
                    if send_val:
                        due_picker = app_due_store.get(app_key)
                        if due_picker:
                            app_guard[app_key] = True
                            try:
                                due_picker.value = send_val + timedelta(days=14)
                            finally:
                                app_guard[app_key] = False

                refresh_all_previews()

            send_picker.observe(_on_send, names="value")

        for app_key, due_picker in app_due_store.items():
            def _on_due(change, app_key=app_key):
                if change.get("name") != "value":
                    return
                if app_guard.get(app_key):
                    return
                app_due_mode[app_key] = "manual"
                refresh_all_previews()

            due_picker.observe(_on_due, names="value")

        def _on_global_send(change):
            if change.get("name") != "value":
                return
            global_send = change.get("new")
            global_due = w_global_due.value

            for app_key in list(app_send_store.keys()):
                send_picker = app_send_store.get(app_key)
                if send_picker and app_send_mode.get(app_key, "auto") == "auto":
                    app_send_guard[app_key] = True
                    try:
                        send_picker.value = global_send
                    finally:
                        app_send_guard[app_key] = False

                if app_due_mode.get(app_key, "auto") == "auto":
                    send_val = send_picker.value if send_picker else None
                    due_val = global_due
                    if due_val is None and send_val:
                        due_val = send_val + timedelta(days=14)
                    due_picker = app_due_store.get(app_key)
                    if due_picker:
                        app_guard[app_key] = True
                        try:
                            due_picker.value = due_val
                        finally:
                            app_guard[app_key] = False

            refresh_all_previews()

        def _on_global_due(change):
            if change.get("name") != "value":
                return
            global_due = change.get("new")
            for app_key, due_picker in app_due_store.items():
                if app_due_mode.get(app_key, "auto") != "auto":
                    continue
                send_picker = app_send_store.get(app_key)
                send_val = send_picker.value if send_picker else None
                due_val = global_due
                if due_val is None and send_val:
                    due_val = send_val + timedelta(days=14)

                app_guard[app_key] = True
                try:
                    due_picker.value = due_val
                finally:
                    app_guard[app_key] = False

            refresh_all_previews()

        if not global_observers_bound["bound"]:
            w_global_send.observe(_on_global_send, names="value")
            w_global_due.observe(_on_global_due, names="value")
            w_body_before_chart.observe(refresh_all_previews, names="value")
            w_body_after_chart.observe(refresh_all_previews, names="value")
            global_observers_bound["bound"] = True

        build_reviewer_cards()
        refresh_all_previews()


    btn_all_new = widgets.Button(description="All New", layout=widgets.Layout(width="90px"))
    btn_all_followup = widgets.Button(description="All Follow-up", layout=widgets.Layout(width="120px"))
    btn_all_skip = widgets.Button(description="All Skip", layout=widgets.Layout(width="90px"))
    btn_all_new.on_click(lambda _: set_all_app_sections("new"))
    btn_all_followup.on_click(lambda _: set_all_app_sections("followup"))
    btn_all_skip.on_click(lambda _: set_all_app_sections("skip"))

    btn_refresh.on_click(render_all)
    btn_send_all.on_click(trigger_batch_send)
    btn_save_defaults.on_click(save_defaults_json)

    render_all()

    top_panel_children = [
        widgets.HTML("<h3>📧 Email Notification Center</h3>"),
        widgets.HTML(
            "<div style='font-size:12px;color:#555'>Classify each fetched app once at the top. Those labels apply to every reviewer email.</div>"
        ),
        widgets.HBox([btn_refresh, btn_send_all]),
        widgets.HTML("<hr style='margin:8px 0'>"),
        widgets.HTML("<b>Send From</b>"),
        w_send_from,
        widgets.HTML("<div style='font-size:11px;color:#777'>Tip: enter a shared mailbox UPN/email you have <i>Send As</i> or <i>Send on behalf</i> rights for.</div>"),
        widgets.HTML("<b>Subject</b>"),
        w_subject_tpl,
        widgets.HTML("<div style='font-size:11px;color:#777'>Optional placeholder: <code>{reviewer_name}</code></div>"),
        widgets.HTML("<b>Global CC</b>"),
        w_cc_global,
        widgets.HTML("<b>Reply-To</b>"),
        w_reply_to,
        widgets.HTML("<hr style='margin:8px 0'>"),
        widgets.HTML("<b>Default Email Content</b>"),
        widgets.HTML("<div style='font-size:11px;color:#777'>Edit the shared email body text here. Preview cards update immediately. Save to JSON if you want to keep the current values.</div>"),
        widgets.HTML("<b>Body Before Chart</b>"),
        w_body_before_chart,
        widgets.HTML("<b>Body After Chart</b>"),
        w_body_after_chart,
        widgets.HBox([btn_save_defaults]),
        defaults_save_status,
    ]
    if defaults_status.value:
        top_panel_children.append(defaults_status)

    top_panel = widgets.VBox(
        top_panel_children,
        layout=widgets.Layout(background_color="#f0f4f8", padding="10px", border="1px solid #ccc", margin="0 0 10px 0"),
    )

    app_panel = widgets.VBox(
        [
            widgets.HTML("<h4 style='margin:0'>App Sections (Global)</h4>"),
            widgets.HBox([btn_all_new, btn_all_followup, btn_all_skip], layout=widgets.Layout(flex_flow="row wrap")),
            widgets.HBox([w_global_send, w_global_due], layout=widgets.Layout(flex_flow="row wrap", align_items="center")),
            widgets.HTML("<div style='font-size:11px;color:#777'>Global dates apply to <b>auto</b> apps only. Manual per-app Send/Due will not be overwritten.</div>"),
            app_selector_box,
        ],
        layout=widgets.Layout(background_color="#fbfcfe", padding="10px", border="1px solid #d8dde6", margin="0 0 12px 0"),
    )

    if USE_WIDGETS:
        display(top_panel, app_panel, reviewer_box)
    else:
        fallback_html = (
            "<h3>Email Notification Center (Static)</h3>"
            "<div style='font-size:12px;color:#555'>Widgets are not available in this environment. Showing a static pending list.</div>"
            + _build_email_fallback_html(df_raw)
        )
        display(HTML(fallback_html))
